# SBF-2: чистый пайплайн для NGC 1380

Компактная рабочая версия `sbf-1.ipynb`: оставлен принятый путь измерения SBF для NGC 1380, тяжёлые исследовательские диагностики и systematic scans убраны.


In [28]:
import sys
print(sys.executable)


[11:22:55] /Users/zuha/Desktop/FKI/4 курс 2025-2026/course_work-SBF/astro_env/bin/python3


In [ ]:
import argparse, numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import builtins
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
from astropy.stats import sigma_clipped_stats
from scipy.ndimage import gaussian_filter
from scipy import ndimage
from scipy.signal import fftconvolve
from astropy.convolution import Gaussian2DKernel, interpolate_replace_nans
from photutils.isophote import Ellipse, EllipseGeometry, build_ellipse_model
from photutils.segmentation import detect_sources, deblend_sources, SegmentationImage, SourceCatalog
import stpsf
from scipy.fft import fft2, fftfreq, fftshift, set_workers
from photutils.background import Background2D, MedianBackground
from astropy.stats import SigmaClip
from astropy.modeling.models import Sersic2D
from astropy.modeling.fitting import LevMarLSQFitter

_orig_print = builtins.print

def print(*args, **kwargs):
    _orig_print(f"[{time.strftime('%H:%M:%S')}]", *args, **kwargs)


# Конфигурация

Эта ячейка задаёт все фиксированные входы и численные настройки для референсного запуска NGC 1380: пути к кадрам, директорию результатов, фотометрические константы, пороги масок, пределы изофот, параметры PSF/SBF, дополнительную residual-mask и цветовые проверки. Code-cell ниже должен быть единственным источником параметров пайплайна; объяснения вынесены сюда, а не в inline-комментарии.


In [ ]:
f150w_path = Path("data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d.fits")
f090w_path = Path("data/NGC 1380/jw03055-o001_t001_nircam_clear-f090w_i2d.fits")

out_dir = f150w_path.parent
stem = f150w_path.stem

ARCSEC2_PER_SR = 2.350443e-11
MJY_SR_TO_JY_PER_ARCSEC2 = 2.350443e-5
AB_ZEROPOINT_JY = 3631.0

USE_ISOPHOTE = True
DUMP_ISOPHOTES = True
USE_BKG = True
DO_DEBLEND = True
DRY = False

FIXED_CENTER = None
CENTER_GUESS_DOWN = 4
CENTER_GUESS_SMOOTH_SIGMA = 3.0
CENTER_GUESS_Q = 99.5
CENTER_GUESS_MIN_PIXELS = 50
CENTER_GUESS_VALID_FLOOR = 1e-6
CENTER_GUESS_WEIGHT_FLOOR = 1e-12

BKG_BOX0 = 256
BKG_BOX2D = 256
BKG_FILTER = 5
SIGMA_STAT = 3.0
SIGMA_DET = 2.5
SIGMA_MAXIT = 5
MASK_NPIXELS = 4
SOURCE_DETECT_CONNECTIVITY = 8
DEBLEND_NLEVELS = 8
DEBLEND_CONTRAST = 0.001
DEBLEND_NPROC = 8
PREMASK_MAX_COMPACT_AREA = 25 * 25
BKG_CHECK_CORNER_FRAC = 0.10
BKG_CHECK_MIN_PIXELS = 100

SERSIC_FIT_SAMPLE_STEP = 16
SERSIC_INIT_PERCENTILE = 95
SERSIC_INIT_REFF_DIV = 8.0
SERSIC_MIN_AMPLITUDE = 1e-6
SERSIC_MIN_REFF = 10.0
SERSIC_INIT_N = 4.0
SERSIC_INIT_ELLIP = 0.2
SERSIC_INIT_THETA = 0.0
SERSIC_REFF_BOUND_MIN = 5.0
SERSIC_N_BOUNDS = (0.5, 8.0)
SERSIC_ELLIP_BOUND_MAX = 0.9

HALF_SIZE = 3000
ISO_START_SMA = 50.0
ISO_START_EPS = 0.2
ISO_START_PA = 0.0
ISO_MAXSMA_FIT = 2000.0
ISO_MINSMA = 15.0
ISO_STEP_MAIN = 10.0
ISO_STEP_COARSE = 20.0
ISO_CHECK_SMA_RANGE = (100.0, 300.0)
ISO_FIT_USE_REAL_PIXELS = True
ISO_FIT_ALLOW_FILLED_FALLBACK = True

SBF_LIT_INNER_ARCSEC = (8.2, 16.4)
SBF_LIT_OUTER_ARCSEC = (16.4, 32.8)
SBF_MODEL_MARGIN = 1.03
MODEL_FULL_CHUNK_ROWS = 256
EXTRAP_GEOM_N = 20
EXTRAP_PROFILE_FIT_N = 20
EXTRAP_PROFILE_FIT_MIN_N = 5
MODEL_GEOM_EPS_MAX = 0.95
MODEL_FULL_EPS_CLIP_MAX = 0.90
MODEL_FULL_Q_MIN = 0.05
MODEL_FULL_BLEND_WIDTH_PX = 80.0
MODEL_FULL_BLEND_MIN_PX = 30.0
MODEL_FULL_BLEND_MAX_PX = 120.0
ISO_REAL_COVERAGE_MIN = 0.50
MODEL_SYNTH_SMA_STEP_PX = 10.0
MODEL_SYNTH_RELAX_SCALE_PX = 250.0
MODEL_CENTER_SLOPE_CLIP = 1.0e-2
MODEL_EPS_SLOPE_CLIP = 3.0e-4
MODEL_PA_SLOPE_CLIP = 3.0e-4

ROBUST_SCALE_FLOOR = 1e-12
GEOM_Q_FLOOR = 1e-3
MIN_PIXELS_ISO_CROP = 5000
MIN_PIXELS_SBF = 5000
MIN_PIXELS_SIGMA_CLIP = 100
MIN_POINTS_PK_BIN = 10
MIN_POINTS_FIT = 10
MIN_COLOR_PIXELS = 100
MIN_WORKING_ISOPHOTES = 10
MIN_ISO_CHECK_POINTS = 5
MIN_ISOPHOTES_MODEL_PROFILE = 8
MIN_ISOPHOTES_MODEL_GEOM = 5

CLIP_SIGMA_QC = 3.5
CLIP_MAXIT_QC = 5
CLIP_TAG_QC = f"{CLIP_SIGMA_QC:g}".replace(".", "p")

WIN_LEN = 8
WIN_STEP = 2
QUALITY_MAD_SCALE = 1.5
PLATEAU_MIN_LEN = 5
PLATEAU_MAX_LEN = None
DM_RANGE_MAX = 0.15
PLATEAU_ERR_WEIGHT = 0.7
PLATEAU_LEN_WEIGHT = 0.5
WINDOW_LEVEL_WEIGHT = 1.0
WINDOW_DYN_WEIGHT = 0.03
WINDOW_STD_WEIGHT = 0.03
WINDOW_MASK_WEIGHT = 0.03
PLATEAU_SPEC_MIN_SUCCESS = 3
SINGLE_WINDOW_SPEC_MIN_SUCCESS = 1

FFT_WORKERS = -1
PSF_SIZE = 129
PSF_NLAMBDA = 7
PSFREF = None
FFT_E_REALIZATIONS_MAIN = 64
FFT_E_REALIZATIONS_DIAG = 64
FFT_KBINS_N = 80
FFT_K_RANGE_MAIN = (0.04, 0.25)
SBF_REGION_K_WINDOWS = [
    (0.01, 0.25),
    (0.03, 0.25),
    FFT_K_RANGE_MAIN,
]
FIT_CORR_WARN = 0.3
FIT_P1_WARN_FACTOR = 5.0

SBF_PR_ENABLE = True
SBF_PR_MAG_BIN = 0.25
SBF_PR_MLIM_OVERRIDE = None
SBF_PR_MLIM_OFFSET = 0.0
SBF_PR_FIT_WIDTH = 1.5
SBF_PR_MIN_SOURCES_REGION = 6
SBF_PR_MIN_SOURCES_GLOBAL = 20
SBF_PR_MIN_FIT_BINS = 3
SBF_PR_GAMMA_BOUNDS = (0.10, 0.70)
SBF_PR_DEFAULT_GAMMA = 0.30
SBF_PR_FALLBACK_QUANTILE = 0.80
SBF_PR_CATALOG_MIN_FLUX_JY = 0.0
SBF_PR_MAX_CORRECTION_WARN = 0.50
SBF_PR_DET_SIGMA = SIGMA_DET
SBF_PR_DET_NPIXELS = MASK_NPIXELS
SBF_PR_MAX_COMPACT_AREA = PREMASK_MAX_COMPACT_AREA
SBF_PR_DO_DEBLEND = DO_DEBLEND
SBF_PR_SOURCE_IMAGE = "img_minus_model_full_unmasked"

COLOR_CLIP_SIGMA = 3.0
COLOR_CLIP_MAXIT = 5

UNROLL_N_BINS = 360
UNROLL_N_SUBANNULI = 6
UNROLL_PAD_EXTRA = 3
UNROLL_MIN_PIXELS = 10
UNROLL_SPLINE_POINTS = 1000
UNROLL_SPLINE_SMOOTH_BINS = 21

FFT_RNG_SEED = 1489


## Загрузка JWST-кадров

Загружаем F150W science frame для SBF и, если файл доступен, F090W frame для цветовых проверок. Здесь же читаются WCS и фотометрические метаданные, которые дальше нужны для pixel scale и AB-калибровки.


In [ ]:
print("loading i2d files...")

with fits.open(f150w_path, memmap=False) as hdul:
    img_f150 = hdul["SCI"].data.astype(float)
    hdr150 = hdul["SCI"].header
    valid150 = np.isfinite(img_f150)

    if "WHT" in hdul:
        try:
            wht150 = hdul["WHT"].data
            valid150 &= np.isfinite(wht150) & (wht150 > 0)
        except Exception:
            pass

wcs150 = WCS(hdr150)
pixar_sr = float(hdr150["PIXAR_SR"])
pix_area = pixar_sr / ARCSEC2_PER_SR
print(f"[WCS] PIXAR_SR={pixar_sr:.8e} sr → pix_area={pix_area:.8e} arcsec^2")

img_f090 = None
hdr090 = None
valid090 = None
wcs090 = None

if f090w_path.exists():
    with fits.open(f090w_path, memmap=False) as hdul:
        img_f090 = hdul["SCI"].data.astype(float)
        hdr090 = hdul["SCI"].header
        valid090 = np.isfinite(img_f090)

        if "WHT" in hdul:
            try:
                wht090 = hdul["WHT"].data
                valid090 &= np.isfinite(wht090) & (wht090 > 0)
            except Exception:
                pass

    wcs090 = WCS(hdr090)

print(f"F150W shape = {img_f150.shape}, valid = {valid150.sum()}")
if img_f090 is not None:
    print(f"F090W shape = {img_f090.shape}, valid = {valid090.sum()}")


## Быстрая оценка центра

Определяем грубый центр галактики по валидной области кадра и ярким пикселям. Этот центр нужен как начальное приближение для Sérsic и изофот, но не является отдельной финальной геометрией SBF.


In [ ]:
def guess_center_fast(img, valid, down=None, sigma=None, q=None, min_sel_pixels=None, wcs=None, log=True):
    if down is None:
        down = CENTER_GUESS_DOWN
    if sigma is None:
        sigma = CENTER_GUESS_SMOOTH_SIGMA
    if q is None:
        q = CENTER_GUESS_Q
    if min_sel_pixels is None:
        min_sel_pixels = CENTER_GUESS_MIN_PIXELS

    down = int(down)
    sigma = float(sigma)
    q = float(q)
    min_sel_pixels = int(min_sel_pixels)

    ny, nx = img.shape

    def _log_center(xc, yc, note=""):
        if not log:
            return
        msg = f"[CENTER-FAST] x={xc:.2f}, y={yc:.2f}"
        if note:
            msg += f" ({note})"
        if wcs is not None:
            ra_deg, dec_deg = wcs.pixel_to_world_values(xc, yc)
            msg += f" | RA={ra_deg:.8f} deg, Dec={dec_deg:.8f} deg"
        print(msg)

    img_d = img[::down, ::down]
    val_d = valid[::down, ::down] & np.isfinite(img_d)

    if not np.any(val_d):
        xc, yc = nx / 2.0, ny / 2.0
        _log_center(xc, yc, note="fallback: no valid downsampled pixels")
        return xc, yc

    data = np.where(val_d, img_d, 0.0).astype(np.float32)
    w = val_d.astype(np.float32)

    num = gaussian_filter(data, sigma=sigma)
    den = gaussian_filter(w, sigma=sigma)
    sm = np.divide(
        num,
        den,
        out=np.full_like(num, np.nan),
        where=den > CENTER_GUESS_VALID_FLOOR,
    )

    if not np.isfinite(sm).any():
        xc, yc = nx / 2.0, ny / 2.0
        _log_center(xc, yc, note="fallback: sm is all-NaN")
        return xc, yc

    thr = np.nanpercentile(sm, q)
    sel = np.isfinite(sm) & (sm >= thr)

    if sel.sum() < min_sel_pixels:
        y, x = np.unravel_index(np.nanargmax(sm), sm.shape)
        xc, yc = float(x * down), float(y * down)
        _log_center(xc, yc, note="fallback: argmax")
        return xc, yc

    ys, xs = np.nonzero(sel)
    ws = sm[sel] - np.nanmin(sm[sel])
    ws = np.nan_to_num(ws, nan=0.0) + CENTER_GUESS_WEIGHT_FLOOR

    x0 = float((xs * ws).sum() / ws.sum()) * down
    y0 = float((ys * ws).sum() / ws.sum()) * down
    _log_center(x0, y0, note=f"down={down}, sigma={sigma}, q={q}")
    return x0, y0


## Подготовка кадра

Готовим science image к моделированию: вычитаем крупномасштабный фон, проверяем остаточный фон и строим первичную маску компактных объектов перед фитированием света галактики.


## Вычитание фона

Оцениваем и вычитаем гладкий фон по валидным пикселям вне яркой центральной области. Полученный F150W кадр используется дальше для центра, Sérsic, изофот и построения residual.


In [ ]:
print("estimating and subtracting background...")

img0 = np.array(img_f150, copy=True)

work0 = np.array(img0, copy=True)
work0[~valid150] = np.nan

if USE_BKG:

    bkg0 = Background2D(
        work0,
        box_size=BKG_BOX0,
        filter_size=(BKG_FILTER, BKG_FILTER),
        sigma_clip=SigmaClip(sigma=SIGMA_STAT, maxiters=SIGMA_MAXIT),
        bkg_estimator=MedianBackground(),
        mask=~valid150,
    )

    bg_scalar = float(np.nanmedian(bkg0.background))
    print(f"[BKG] rough scalar background = {bg_scalar:.3e}")

    img1 = img0 - bg_scalar

    det_work = np.array(img1, copy=True)
    det_work[~valid150] = np.nan

    mean_det, med_det, std_det = sigma_clipped_stats(
        det_work[np.isfinite(det_work)],
        sigma=SIGMA_STAT,
        maxiters=SIGMA_MAXIT,
    )

    thr_det = med_det + SIGMA_DET * max(std_det, ROBUST_SCALE_FLOOR)

    segm = detect_sources(
        det_work,
        threshold=thr_det,
        npixels=MASK_NPIXELS,
        connectivity=SOURCE_DETECT_CONNECTIVITY,
    )

    src_mask = np.zeros_like(img1, dtype=bool)
    if segm is not None:
        src_mask = segm.data > 0

    final_mask = (~valid150) | src_mask

    bkg2d = Background2D(
        img1,
        box_size=BKG_BOX2D,
        filter_size=(BKG_FILTER, BKG_FILTER),
        sigma_clip=SigmaClip(sigma=SIGMA_STAT, maxiters=SIGMA_MAXIT),
        bkg_estimator=MedianBackground(),
        mask=final_mask,
    )

    bg_map = np.array(bkg2d.background, copy=True)
    img = img1 - bg_map

    print(
        f"[BKG] final map: "
        f"min={np.nanmin(bg_map):.3e}, "
        f"med={np.nanmedian(bg_map):.3e}, "
        f"max={np.nanmax(bg_map):.3e}"
    )

else:
    bg_scalar = 0.0
    bg_map = np.zeros_like(img0)
    img = img0.copy()
    print("[BKG] skipped")


## Проверка фона

Измеряем остаточный фон после вычитания во внешней валидной области. Это быстрая проверка, которая должна поймать грубую ошибку sky-level до дорогих ячеек модели.


In [ ]:
print("checking residual background after subtraction...")

check = np.array(img, copy=True)
check[~valid150] = np.nan

ny, nx = check.shape
dx = max(1, int(nx * BKG_CHECK_CORNER_FRAC))
dy = max(1, int(ny * BKG_CHECK_CORNER_FRAC))

corners = np.concatenate([
    check[:dy, :dx].ravel(),
    check[:dy, -dx:].ravel(),
    check[-dy:, :dx].ravel(),
    check[-dy:, -dx:].ravel(),
])

corners = corners[np.isfinite(corners)]

if corners.size > BKG_CHECK_MIN_PIXELS:
    mean_c, med_c, std_c = sigma_clipped_stats(corners, sigma=SIGMA_STAT, maxiters=SIGMA_MAXIT)
    print(
        f"[BKG-CHECK] corners: "
        f"med={med_c:.3e}, mean={mean_c:.3e}, std={std_c:.3e}, N={corners.size}"
    )
else:
    print("[BKG-CHECK] too few valid corner pixels")


## Первичная маска компактных источников

Детектируем яркие компактные источники на кадре после вычитания фона. Маска защищает Sérsic и изофоты от звёзд, шаровиков и компактных contaminant sources, но должна сохранять основной свет галактики.


In [ ]:
use_det = valid150 & np.isfinite(img)

mean_det, med_det, std_det = sigma_clipped_stats(
    img[use_det],
    sigma=SIGMA_STAT,
    maxiters=SIGMA_MAXIT,
)

thr_det = med_det + SIGMA_DET * max(std_det, ROBUST_SCALE_FLOOR)
print(f"[PREMASK] med={med_det:.3e}, std={std_det:.3e}, threshold={thr_det:.3e}")

segm = detect_sources(
    img,
    threshold=thr_det,
    npixels=MASK_NPIXELS,
    connectivity=SOURCE_DETECT_CONNECTIVITY,
)
print(f"[PREMASK] detected {segm.nlabels if segm is not None else 0} sources")

compact_labels = []
premask_segm = segm

if segm is None:
    premask_src = np.zeros_like(img, dtype=bool)
else:
    if DO_DEBLEND:
        print("[PREMASK] deblending sources...")
        try:
            segm = deblend_sources(
                img,
                segm,
                npixels=MASK_NPIXELS,
                nlevels=DEBLEND_NLEVELS,
                contrast=DEBLEND_CONTRAST,
                nproc=DEBLEND_NPROC,
                progress_bar=False,
            )
        except Exception as e:
            print(f"[PREMASK] deblend skipped: {e}")

    premask_segm = segm

    max_compact_area = PREMASK_MAX_COMPACT_AREA
    labels = segm.labels
    counts = np.bincount(segm.data.ravel(), minlength=labels.max() + 1)
    counts[0] = 0

    compact_labels = [int(lab) for lab in labels if 0 < counts[lab] <= max_compact_area]

    premask_src = np.zeros_like(img, dtype=bool)
    for lab in compact_labels:
        premask_src |= (segm.data == lab)

premask_compact_labels = np.array(compact_labels, dtype=int)

premask = (~valid150) | premask_src

print(f"[PREMASK] compact labels kept = {len(premask_compact_labels)}")
print(f"[PREMASK] coverage = {100.0 * premask.sum() / premask.size:.2f}%")


## Принятый центр галактики

Выбираем рабочий центр для инициализации гладкой модели. Используется быстрая оценка, если в конфиге явно не задан ручной центр.


In [ ]:
print("choosing galaxy center...")

if FIXED_CENTER is None:
    x0_center, y0_center = guess_center_fast(
        img,
        valid150 & (~premask),
        down=CENTER_GUESS_DOWN,
        sigma=CENTER_GUESS_SMOOTH_SIGMA,
        q=CENTER_GUESS_Q,
        min_sel_pixels=CENTER_GUESS_MIN_PIXELS,
        wcs=wcs150,
        log=True,
    )
    center_src = "auto-fast"
else:
    x0_center, y0_center = FIXED_CENTER
    ra_deg, dec_deg = wcs150.pixel_to_world_values(x0_center, y0_center)
    print(f"[CENTER-FIXED] x={x0_center:.2f}, y={y0_center:.2f} | RA={ra_deg:.8f} deg, Dec={dec_deg:.8f} deg")
    center_src = "fixed"

print(f"[CENTER] using ({x0_center:.2f}, {y0_center:.2f}) [{center_src}]")


## Sérsic fallback-модель

Фитируем простую гладкую Sérsic-компоненту по замаскированному кадру. В текущем science path это в основном fallback/fill для невалидных пикселей, а не финальная модель для вычитания SBF.


In [ ]:
yy_full, xx_full = np.indices(img.shape, dtype=float)

use_fit = (~premask) & np.isfinite(img)

sample_step = SERSIC_FIT_SAMPLE_STEP

y_fit = yy_full[use_fit][::sample_step]
x_fit = xx_full[use_fit][::sample_step]
z_fit = img[use_fit][::sample_step]

print(f"[SERSIC] fit sample size = {z_fit.size}")

amp0 = float(np.nanpercentile(z_fit, SERSIC_INIT_PERCENTILE))

r_eff0 = float(min(img.shape) / SERSIC_INIT_REFF_DIV)

sersic_init = Sersic2D(

    amplitude=max(amp0, SERSIC_MIN_AMPLITUDE),

    r_eff=max(r_eff0, SERSIC_MIN_REFF),

    n=SERSIC_INIT_N,

    x_0=float(x0_center),
    y_0=float(y0_center),

    ellip=SERSIC_INIT_ELLIP,

    theta=SERSIC_INIT_THETA,
)

sersic_init.amplitude.bounds = (0.0, None)

sersic_init.r_eff.bounds = (SERSIC_REFF_BOUND_MIN, float(max(img.shape)))

sersic_init.n.bounds = SERSIC_N_BOUNDS

sersic_init.x_0.bounds = (0.0, float(img.shape[1] - 1))
sersic_init.y_0.bounds = (0.0, float(img.shape[0] - 1))

sersic_init.ellip.bounds = (0.0, SERSIC_ELLIP_BOUND_MAX)

sersic_init.theta.bounds = (-np.pi / 2.0, np.pi / 2.0)

fitter = LevMarLSQFitter()

try:
    sersic_fit = fitter(sersic_init, x_fit, y_fit, z_fit)
    print(
        "[SERSIC] fitted:",
        f"amp={float(sersic_fit.amplitude.value):.3e},",
        f"r_eff={float(sersic_fit.r_eff.value):.2f},",
        f"n={float(sersic_fit.n.value):.2f},",
        f"x0={float(sersic_fit.x_0.value):.2f},",
        f"y0={float(sersic_fit.y_0.value):.2f},",
        f"ellip={float(sersic_fit.ellip.value):.3f},",
        f"theta={float(sersic_fit.theta.value):.3f}",
    )
except Exception as e:
    print(f"[SERSIC] fit failed, fallback to initial model: {e}")
    sersic_fit = sersic_init

sersic_model = sersic_fit(xx_full, yy_full)

sm = np.array(img, copy=True)

bad = (~valid150) | (~np.isfinite(sm))

sm[bad] = sersic_model[bad]


## Кроп вокруг галактики

Строим cutout вокруг NGC 1380, чтобы изофоты считались быстрее и устойчивее. Координаты crop сохраняются, чтобы затем перенести модель обратно на полный detector frame.


In [ ]:
print("building cutout...")

ny, nx = sm.shape
x1 = max(0, int(x0_center - HALF_SIZE))
x2 = min(nx, int(x0_center + HALF_SIZE))
y1 = max(0, int(y0_center - HALF_SIZE))
y2 = min(ny, int(y0_center + HALF_SIZE))

img_real_c = np.asarray(img[y1:y2, x1:x2], dtype=float)
img_fill_c = np.asarray(sm[y1:y2, x1:x2], dtype=float)
img_c = np.array(img_real_c, copy=True)
valid_c = valid150[y1:y2, x1:x2]
mask_c = premask_src[y1:y2, x1:x2]

x0c = x0_center - x1
y0c = y0_center - y1

print(f"[CUTOUT] bounds = x[{x1}:{x2}], y[{y1}:{y2}]")
print(f"[CUTOUT] center in crop = ({x0c:.2f}, {y0c:.2f})")
print(f"[CUTOUT] shape = {img_c.shape}")
print(f"[CUTOUT] real finite = {int(np.isfinite(img_real_c).sum())}, filled finite = {int(np.isfinite(img_fill_c).sum())}")


## Входные данные для изофот

Готовим cutout и маски для `photutils.isophote`. Реальные пиксели детектора остаются основной областью фита; заполненные значения нужны только там, где численному алгоритму требуется конечное изображение.


In [ ]:
print("preparing data for isophote fit...")

data_real = np.asarray(img_real_c, dtype=float).copy()
data_real[(~valid_c) | mask_c] = np.nan
ok_real = np.isfinite(data_real)
data_real_ma = np.ma.array(data_real, mask=~ok_real)

data_fill = np.asarray(img_fill_c, dtype=float).copy()
data_fill[mask_c] = np.nan
ok_fill = np.isfinite(data_fill)
data_fill_ma = np.ma.array(data_fill, mask=~ok_fill)

iso_fit_primary_mode = "real_only" if ISO_FIT_USE_REAL_PIXELS else "filled_primary"
iso_fit_fallback_enabled = bool(ISO_FIT_ALLOW_FILLED_FALLBACK)

print(f"[ISO] real finite pixels in crop = {int(ok_real.sum())}")
print(f"[ISO] filled finite pixels in crop = {int(ok_fill.sum())}")
print(f"[ISO] masked by source mask = {mask_c.sum()}, old invalid in crop = {(~valid_c).sum()}")
print(f"[ISO] primary fit mode = {iso_fit_primary_mode}, filled fallback enabled = {iso_fit_fallback_enabled}")

if ISO_FIT_USE_REAL_PIXELS and ok_real.sum() < MIN_PIXELS_ISO_CROP and not ISO_FIT_ALLOW_FILLED_FALLBACK:
    raise RuntimeError(f"[ISO] too few valid real-data pixels in crop for real-only fit: N={int(ok_real.sum())}")
if (not ISO_FIT_USE_REAL_PIXELS) and ok_fill.sum() < MIN_PIXELS_ISO_CROP:
    raise RuntimeError(f"[ISO] too few valid filled pixels in crop for isophote fit: N={int(ok_fill.sum())}")


## Фит изофот

Фитируем галактику свободными эллиптическими изофотами с плавающим центром и геометрией до заданного предела сходимости. Это основной smooth-light model для SBF residual.


In [ ]:
print("fitting isophotes...")

geom = EllipseGeometry(
    x0=x0c,
    y0=y0c,
    sma=ISO_START_SMA,
    eps=ISO_START_EPS,
    pa=ISO_START_PA,
)

minsma = ISO_MINSMA
fit_maxsma = ISO_MAXSMA_FIT

def _fit_isolist_for_dataset(data_ma_try):
    ell = Ellipse(data_ma_try, geom)
    last_err_local = None
    fit_signature_local = None
    isolist_local = None

    try:
        isolist_local = ell.fit_image(
            minsma=minsma,
            maxsma=fit_maxsma,
            step=ISO_STEP_MAIN,
            linear=True,
        )
        fit_signature_local = "linear=True"
    except TypeError as e:
        last_err_local = e
        print(f"[ISO] fit_image(linear=True) signature mismatch: {e}")
    except Exception as e:
        last_err_local = e
        print(f"[ISO] fit_image(linear=True) failed: {e}")

    if isolist_local is None:
        try:
            isolist_local = ell.fit_image(
                minsma=minsma,
                maxsma=fit_maxsma,
            )
            fit_signature_local = "no linear"
        except Exception as e:
            last_err_local = e
            print(f"[ISO] fit_image(no linear) failed: {e}")

    if isolist_local is None:
        try:
            isolist_local = ell.fit_image(
                minsma=minsma,
                maxsma=fit_maxsma,
                step=ISO_STEP_COARSE,
            )
            fit_signature_local = "coarse step=20"
        except Exception as e:
            last_err_local = e
            print(f"[ISO] fit_image(coarse) failed: {e}")

    return isolist_local, fit_signature_local, last_err_local

fit_datasets = []
if ISO_FIT_USE_REAL_PIXELS:
    fit_datasets.append(("real_only", data_real_ma, int(ok_real.sum())))
    if ISO_FIT_ALLOW_FILLED_FALLBACK:
        fit_datasets.append(("filled_fallback", data_fill_ma, int(ok_fill.sum())))
else:
    fit_datasets.append(("filled_primary", data_fill_ma, int(ok_fill.sum())))

isolist = None
last_err = None
iso_fit_mode_used = None
iso_fit_signature_used = None
fit_attempt_summaries = []

for mode_label, data_ma_try, n_valid_try in fit_datasets:
    print(f"[ISO] trying dataset={mode_label}, N_finite={n_valid_try}")
    if n_valid_try < MIN_PIXELS_ISO_CROP:
        msg = f"[ISO] skipping dataset={mode_label}: too few finite pixels N={n_valid_try}"
        print(msg)
        fit_attempt_summaries.append(msg)
        continue

    isolist_try, fit_signature_try, last_err_try = _fit_isolist_for_dataset(data_ma_try)
    if isolist_try is not None and len(isolist_try) >= MIN_WORKING_ISOPHOTES:
        isolist = isolist_try
        last_err = last_err_try
        iso_fit_mode_used = mode_label
        iso_fit_signature_used = fit_signature_try
        break

    msg = (
        f"[ISO] dataset={mode_label} did not yield a working isolist: "
        f"N={0 if isolist_try is None else len(isolist_try)}, last_err={last_err_try}"
    )
    print(msg)
    fit_attempt_summaries.append(msg)
    last_err = last_err_try

if isolist is None or len(isolist) < MIN_WORKING_ISOPHOTES:
    summary_text = " | ".join(fit_attempt_summaries) if fit_attempt_summaries else "no fit attempts recorded"
    raise RuntimeError(
        f"[ISO] isolist too short after all datasets: {0 if isolist is None else len(isolist)}; "
        f"last_err={last_err}; attempts={summary_text}"
    )

print(f"[ISO] fit_image maxsma = {fit_maxsma}")
print(f"[ISO] fit dataset used = {iso_fit_mode_used}, solver path = {iso_fit_signature_used}")
print(f"[ISO] isolist N={len(isolist)}, maxsma≈{float(isolist[-1].sma):.1f} px")

x0_fit = float(np.nanmedian([iso.x0 for iso in isolist]))
y0_fit = float(np.nanmedian([iso.y0 for iso in isolist]))
eps_fit = float(np.nanmedian([iso.eps for iso in isolist]))
pa_fit = float(np.nanmedian([iso.pa for iso in isolist]))
print(f"[ISO] fitted center≈({x0_fit:.1f}, {y0_fit:.1f}) in crop, eps≈{eps_fit:.3f}, pa≈{pa_fit:.3f} rad")


## Изофотная модель на crop

Строим модель по сходящимся изофотам на crop и локальный residual. Это прямой результат фита до переноса и продолжения модели на полный кадр.


In [ ]:
print("building isophote model...")

model_c = build_ellipse_model(img_real_c.shape, isolist)
model_c = np.where(np.isfinite(model_c) & (model_c > 0.0), model_c, np.nan)

model = np.full_like(img, np.nan)

resid = np.full_like(img, np.nan)

model[y1:y2, x1:x2] = model_c

resid[y1:y2, x1:x2] = img_real_c - model_c

resid[premask] = np.nan

finite_resid = np.isfinite(resid)
print(
    f"[CHK] resid: finite={finite_resid.sum()}, "
    f"min={np.nanmin(resid):.3e}, med={np.nanmedian(resid):.3e}, max={np.nanmax(resid):.3e}"
)

finite_model_c = np.isfinite(model_c)
print(
    f"[CHK] model_c: finite={finite_model_c.sum()}, "
    f"min={np.nanmin(model_c):.3e}, med={np.nanmedian(model_c):.3e}, max={np.nanmax(model_c):.3e}"
)


## Full-frame изофотная модель

Продолжаем семейство изофот на полный detector frame. Результат `model_full` является гладкой моделью галактики для science residual; построение непрерывно связано с fitted isophotes и ограничено заданным радиусом модели.


In [ ]:
from photutils.isophote import IsophoteList

print("building continuous 2D full-frame isophotal model for Jensen/TRGB-SBF annuli...")

pix_scale = float(np.sqrt(pix_area))
r1_in, r1_out = SBF_LIT_INNER_ARCSEC[0] / pix_scale, SBF_LIT_INNER_ARCSEC[1] / pix_scale
r2_in, r2_out = SBF_LIT_OUTER_ARCSEC[0] / pix_scale, SBF_LIT_OUTER_ARCSEC[1] / pix_scale
lit_outer_r_px = float(r2_out)

class _SyntheticIso:
    __slots__ = ("sma", "intens", "eps", "pa", "x0", "y0", "grad", "a3", "b3", "a4", "b4")

    def __init__(self, sma, intens, eps, pa, x0, y0, grad):
        self.sma = float(sma)
        self.intens = float(intens)
        self.eps = float(eps)
        self.pa = float(pa)
        self.x0 = float(x0)
        self.y0 = float(y0)
        self.grad = float(grad)
        self.a3 = 0.0
        self.b3 = 0.0
        self.a4 = 0.0
        self.b4 = 0.0

def _clip_slope(value, limit):
    try:
        value = float(value)
    except Exception:
        return 0.0
    if not np.isfinite(value):
        return 0.0
    return float(np.clip(value, -abs(limit), abs(limit)))

def _weighted_line_fit(x, y, w):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    good = np.isfinite(x) & np.isfinite(y) & np.isfinite(w) & (w > 0.0)
    x = x[good]
    y = y[good]
    w = w[good]
    if x.size == 0:
        return np.nan, 0.0
    if x.size == 1:
        return float(y[0]), 0.0
    try:
        slope, intercept = np.polyfit(x, y, 1, w=w)
    except Exception:
        slope, intercept = np.polyfit(x, y, 1)
    return float(intercept), float(slope)

def _saturating_extension(delta, slope, scale):
    delta = np.asarray(delta, dtype=float)
    if scale <= 0.0:
        return slope * delta
    return slope * scale * (1.0 - np.exp(-delta / scale))

def _gradient_from_profile(sma, intens):
    sma = np.asarray(sma, dtype=float)
    intens = np.asarray(intens, dtype=float)
    if sma.size == 1:
        return np.array([-abs(intens[0]) / max(sma[0], 1.0)], dtype=float)
    grad = np.gradient(intens, sma, edge_order=1)
    bad = (~np.isfinite(grad)) | (grad >= 0.0)
    if np.any(bad):
        safe = np.zeros_like(grad)
        safe[:] = -np.maximum(np.abs(intens), 1e-6) / np.maximum(sma, 1.0)
        grad[bad] = safe[bad]
    return grad

iso_rows = []
for iso in isolist:
    sma_i = float(iso.sma)
    intens_i = float(getattr(iso, "intens", np.nan))
    x0_i = float(iso.x0)
    y0_i = float(iso.y0)
    eps_i = float(iso.eps)
    pa_i = float(iso.pa)
    grad_i = float(getattr(iso, "grad", np.nan))

    real_data_frac = np.nan
    sampled_real = 0
    sampled_total = 0
    try:
        xs, ys = iso.sampled_coordinates()
        xi = np.rint(xs).astype(int)
        yi = np.rint(ys).astype(int)
        inside = (xi >= 0) & (xi < valid_c.shape[1]) & (yi >= 0) & (yi < valid_c.shape[0])
        sampled_total = int(np.count_nonzero(inside))
        if sampled_total > 0:
            sampled_real = int(np.count_nonzero(valid_c[yi[inside], xi[inside]]))
            real_data_frac = float(sampled_real / sampled_total)
    except Exception:
        pass

    iso_rows.append({
        "sma": sma_i,
        "intens": intens_i,
        "x0_crop": x0_i,
        "y0_crop": y0_i,
        "x0_full": float(x1 + x0_i),
        "y0_full": float(y1 + y0_i),
        "eps": eps_i,
        "pa": pa_i,
        "grad": grad_i,
        "real_data_frac": real_data_frac,
        "sampled_real": sampled_real,
        "sampled_total": sampled_total,
    })

iso_df = pd.DataFrame(iso_rows).sort_values("sma").reset_index(drop=True)
iso_df["pa_unwrap"] = np.unwrap(iso_df["pa"].to_numpy(dtype=float))

fit_good = (
    np.isfinite(iso_df["sma"])
    & np.isfinite(iso_df["intens"])
    & np.isfinite(iso_df["x0_full"])
    & np.isfinite(iso_df["y0_full"])
    & np.isfinite(iso_df["eps"])
    & np.isfinite(iso_df["pa_unwrap"])
    & (iso_df["sma"] > 0.0)
    & (iso_df["intens"] > 0.0)
    & (iso_df["eps"] >= 0.0)
    & (iso_df["eps"] < MODEL_GEOM_EPS_MAX)
)
fit_df = iso_df.loc[fit_good].copy().reset_index(drop=True)
if len(fit_df) < MIN_ISOPHOTES_MODEL_PROFILE:
    raise RuntimeError(f"[MODEL-FULL] too few valid fitted isophotes for 2D family: N={len(fit_df)}")

fit_df = fit_df.drop_duplicates(subset="sma", keep="last").sort_values("sma").reset_index(drop=True)

sma_profile = fit_df["sma"].to_numpy(dtype=float)
int_profile = fit_df["intens"].to_numpy(dtype=float)
fitted_sma_min_px = float(np.nanmin(sma_profile))
fitted_sma_max_px = float(np.nanmax(sma_profile))

real_cov = fit_df["real_data_frac"].to_numpy(dtype=float)
real_cov_use = np.isfinite(real_cov) & (real_cov >= ISO_REAL_COVERAGE_MIN)
if np.any(real_cov_use):
    trusted_sma_max_px = float(np.nanmax(fit_df.loc[real_cov_use, "sma"]))
else:
    trusted_sma_max_px = fitted_sma_max_px

x0_model_full = float(np.nanmedian(fit_df["x0_full"]))
y0_model_full = float(np.nanmedian(fit_df["y0_full"]))

edge_row = fit_df.iloc[-1]
sma_edge = float(edge_row["sma"])
I_at_fit_edge = float(edge_row["intens"])
x0_edge = float(edge_row["x0_full"])
y0_edge = float(edge_row["y0_full"])
eps_edge = float(np.clip(edge_row["eps"], 0.0, MODEL_FULL_EPS_CLIP_MAX))
pa_edge = float(edge_row["pa_unwrap"])

fit_tail_n = min(EXTRAP_GEOM_N, len(fit_df))
tail_df = fit_df.tail(fit_tail_n).copy()
tail_w = np.clip(np.nan_to_num(tail_df["real_data_frac"].to_numpy(dtype=float), nan=0.25), 0.05, 1.0)
tail_sma = tail_df["sma"].to_numpy(dtype=float)

_, x0_slope = _weighted_line_fit(tail_sma, tail_df["x0_full"].to_numpy(dtype=float), tail_w)
_, y0_slope = _weighted_line_fit(tail_sma, tail_df["y0_full"].to_numpy(dtype=float), tail_w)
_, eps_slope = _weighted_line_fit(tail_sma, tail_df["eps"].to_numpy(dtype=float), tail_w)
_, pa_slope = _weighted_line_fit(tail_sma, tail_df["pa_unwrap"].to_numpy(dtype=float), tail_w)

x0_slope = _clip_slope(x0_slope, MODEL_CENTER_SLOPE_CLIP)
y0_slope = _clip_slope(y0_slope, MODEL_CENTER_SLOPE_CLIP)
eps_slope = _clip_slope(eps_slope, MODEL_EPS_SLOPE_CLIP)
pa_slope = _clip_slope(pa_slope, MODEL_PA_SLOPE_CLIP)

prof_tail_n = min(EXTRAP_PROFILE_FIT_N, len(fit_df))
prof_tail = fit_df.tail(prof_tail_n).copy()
prof_tail_w = np.clip(np.nan_to_num(prof_tail["real_data_frac"].to_numpy(dtype=float), nan=0.25), 0.05, 1.0)
outer_log_slope = 0.0
profile_extrap_note = "flat outer fallback"
if prof_tail_n >= EXTRAP_PROFILE_FIT_MIN_N and np.all(prof_tail["intens"].to_numpy(dtype=float) > 0.0):
    _, outer_log_slope = _weighted_line_fit(
        prof_tail["sma"].to_numpy(dtype=float),
        np.log(prof_tail["intens"].to_numpy(dtype=float)),
        prof_tail_w,
    )
    if np.isfinite(outer_log_slope) and outer_log_slope < 0.0:
        profile_extrap_note = f"log-linear outer tail, dlnI/dsma={outer_log_slope:.3e} pix^-1"
    else:
        outer_log_slope = 0.0

relax_scale_px = float(max(MODEL_SYNTH_RELAX_SCALE_PX, MODEL_SYNTH_SMA_STEP_PX))
max_delta_probe = max(lit_outer_r_px, relax_scale_px)
eps_outer_probe = float(np.clip(
    eps_edge + _saturating_extension(max_delta_probe, eps_slope, relax_scale_px),
    0.0,
    MODEL_FULL_EPS_CLIP_MAX,
))
q_outer_probe = max(MODEL_FULL_Q_MIN, 1.0 - eps_outer_probe)
sma_needed_for_outer_circle_px = float(lit_outer_r_px / q_outer_probe)
sma_model_full_max_px = float(max(fitted_sma_max_px, sma_needed_for_outer_circle_px) * SBF_MODEL_MARGIN)

sma_step_syn = float(max(MODEL_SYNTH_SMA_STEP_PX, 1.0))
outer_sma = np.arange(fitted_sma_max_px + sma_step_syn, sma_model_full_max_px + 0.5 * sma_step_syn, sma_step_syn)
if outer_sma.size and outer_sma[-1] < sma_model_full_max_px:
    outer_sma = np.append(outer_sma, sma_model_full_max_px)
elif not outer_sma.size and sma_model_full_max_px > fitted_sma_max_px + 0.5:
    outer_sma = np.array([sma_model_full_max_px], dtype=float)

delta_outer = np.maximum(outer_sma - fitted_sma_max_px, 0.0)
if outer_sma.size:
    x0_syn = x0_edge + _saturating_extension(delta_outer, x0_slope, relax_scale_px)
    y0_syn = y0_edge + _saturating_extension(delta_outer, y0_slope, relax_scale_px)
    eps_syn = np.clip(eps_edge + _saturating_extension(delta_outer, eps_slope, relax_scale_px), 0.0, MODEL_FULL_EPS_CLIP_MAX)
    pa_syn = pa_edge + _saturating_extension(delta_outer, pa_slope, relax_scale_px)
    int_syn = I_at_fit_edge * np.exp(outer_log_slope * delta_outer)
else:
    x0_syn = y0_syn = eps_syn = pa_syn = int_syn = np.array([], dtype=float)

fit_nodes = fit_df[["sma", "intens", "x0_full", "y0_full", "eps", "pa_unwrap"]].copy()
fit_nodes = fit_nodes.rename(columns={"pa_unwrap": "pa"})
fit_nodes["node_origin"] = "fitted"

if outer_sma.size:
    syn_nodes = pd.DataFrame({
        "sma": outer_sma,
        "intens": int_syn,
        "x0_full": x0_syn,
        "y0_full": y0_syn,
        "eps": eps_syn,
        "pa": pa_syn,
        "node_origin": "synthetic",
    })
    family_nodes = pd.concat([fit_nodes, syn_nodes], ignore_index=True)
else:
    family_nodes = fit_nodes.copy()

family_nodes = family_nodes.drop_duplicates(subset="sma", keep="last").sort_values("sma").reset_index(drop=True)
family_nodes = family_nodes[np.isfinite(family_nodes["intens"]) & (family_nodes["intens"] > 0.0)].reset_index(drop=True)
family_nodes["grad"] = _gradient_from_profile(
    family_nodes["sma"].to_numpy(dtype=float),
    family_nodes["intens"].to_numpy(dtype=float),
)
fit_nodes["grad"] = _gradient_from_profile(
    fit_nodes["sma"].to_numpy(dtype=float),
    fit_nodes["intens"].to_numpy(dtype=float),
)

fit_iso_full = IsophoteList([
    _SyntheticIso(row.sma, row.intens, row.eps, row.pa, row.x0_full, row.y0_full, row.grad)
    for row in fit_nodes.itertuples(index=False)
])
family_iso_full = IsophoteList([
    _SyntheticIso(row.sma, row.intens, row.eps, row.pa, row.x0_full, row.y0_full, row.grad)
    for row in family_nodes.itertuples(index=False)
])

model_fit_full = build_ellipse_model(img.shape, fit_iso_full, fill=np.nan)
model_fit_full = np.where(np.isfinite(model_fit_full) & (model_fit_full > 0.0), model_fit_full, np.nan)

model_full = build_ellipse_model(img.shape, family_iso_full, fill=np.nan)
model_full = np.where(np.isfinite(model_full) & (model_full > 0.0), model_full, np.nan)
model_full[(~valid150) | (~np.isfinite(img))] = np.nan
model_fit_full[(~valid150) | (~np.isfinite(img))] = np.nan

resid_full = np.array(img - model_full, dtype=np.float32, copy=True)
resid_full[premask | (~np.isfinite(model_full)) | (~np.isfinite(img))] = np.nan

fitted_support_mask = np.isfinite(model_fit_full) & (model_fit_full > 0.0)
synthetic_support_mask = np.isfinite(model_full) & (model_full > 0.0) & (~fitted_support_mask)
n_fitted_support = int(fitted_support_mask.sum())
n_cont_only = int(synthetic_support_mask.sum())

x0_sbf_circ = x0_model_full
y0_sbf_circ = y0_model_full

def circular_annulus_model_coverage(r_in, r_out):
    yy_full, xx_full = np.indices(img.shape, dtype=float)
    rr = np.hypot(xx_full - x0_sbf_circ, yy_full - y0_sbf_circ)
    ann = (rr >= float(r_in)) & (rr <= float(r_out))
    valid_ann = ann & valid150 & np.isfinite(img)
    model_ann = valid_ann & np.isfinite(model_full) & (model_full > 0.0)
    fitted_ann = model_ann & fitted_support_mask
    cont_ann = model_ann & synthetic_support_mask
    unmasked_model_ann = model_ann & (~premask)
    total_valid = int(valid_ann.sum())
    model_valid = int(model_ann.sum())
    fitted_valid = int(fitted_ann.sum())
    continuation_valid = int(cont_ann.sum())
    unmasked_model_valid = int(unmasked_model_ann.sum())
    frac_model = float(model_valid / total_valid) if total_valid > 0 else np.nan
    frac_unmasked_model = float(unmasked_model_valid / total_valid) if total_valid > 0 else np.nan
    frac_fitted = float(fitted_valid / total_valid) if total_valid > 0 else np.nan
    frac_cont = float(continuation_valid / total_valid) if total_valid > 0 else np.nan
    return total_valid, model_valid, unmasked_model_valid, frac_model, frac_unmasked_model, fitted_valid, continuation_valid, frac_fitted, frac_cont

inner_cov = circular_annulus_model_coverage(r1_in, r1_out)
outer_cov = circular_annulus_model_coverage(r2_in, r2_out)

print(
    f"[MODEL-FULL] fitted profile sma range = "
    f"{fitted_sma_min_px:.1f}..{fitted_sma_max_px:.1f} px "
    f"({fitted_sma_min_px * pix_scale:.2f}..{fitted_sma_max_px * pix_scale:.2f} arcsec)"
)
print(
    f"[MODEL-FULL] Jensen/TRGB-SBF circular annuli = "
    f"{SBF_LIT_INNER_ARCSEC[0]:.1f}-{SBF_LIT_INNER_ARCSEC[1]:.1f} arcsec and "
    f"{SBF_LIT_OUTER_ARCSEC[0]:.1f}-{SBF_LIT_OUTER_ARCSEC[1]:.1f} arcsec "
    f"= {r1_in:.1f}-{r1_out:.1f} px and {r2_in:.1f}-{r2_out:.1f} px"
)
print(
    f"[MODEL-FULL] trusted real-data support extends to sma≈{trusted_sma_max_px:.1f} px "
    f"({trusted_sma_max_px * pix_scale:.2f} arcsec) at real-data coverage >= {ISO_REAL_COVERAGE_MIN:.2f}"
)
print(
    f"[MODEL-FULL] fitted center median: x0={x0_model_full:.2f}, y0={y0_model_full:.2f}; "
    f"outer edge geometry: eps={eps_edge:.3f}, q={max(MODEL_FULL_Q_MIN, 1.0 - eps_edge):.3f}, pa={pa_edge:.3f} rad"
)
print(
    f"[MODEL-FULL] outer continuation slopes: dx0/dsma={x0_slope:.3e}, dy0/dsma={y0_slope:.3e}, "
    f"deps/dsma={eps_slope:.3e}, dpa/dsma={pa_slope:.3e}; relax_scale={relax_scale_px:.1f} px"
)
print(
    f"[MODEL-FULL] to cover circular R=32.8 arcsec, required elliptical sma≈"
    f"{sma_needed_for_outer_circle_px:.1f} px; built to {sma_model_full_max_px:.1f} px"
)
print(
    f"[MODEL-FULL] node counts: fitted={len(fit_nodes)}, synthetic={len(family_nodes) - len(fit_nodes)}, "
    f"family_total={len(family_nodes)}"
)
print(f"[MODEL-FULL] outer profile extrapolation: {profile_extrap_note}")
print(
    f"[MODEL-FULL] inner annulus model coverage: "
    f"valid={inner_cov[0]}, model={inner_cov[1]} ({100.0 * inner_cov[3]:.2f}%), "
    f"fitted={inner_cov[5]} ({100.0 * inner_cov[7]:.2f}%), "
    f"continuation={inner_cov[6]} ({100.0 * inner_cov[8]:.2f}%), "
    f"unmasked+model={inner_cov[2]} ({100.0 * inner_cov[4]:.2f}%)"
)
print(
    f"[MODEL-FULL] outer annulus model coverage: "
    f"valid={outer_cov[0]}, model={outer_cov[1]} ({100.0 * outer_cov[3]:.2f}%), "
    f"fitted={outer_cov[5]} ({100.0 * outer_cov[7]:.2f}%), "
    f"continuation={outer_cov[6]} ({100.0 * outer_cov[8]:.2f}%), "
    f"unmasked+model={outer_cov[2]} ({100.0 * outer_cov[4]:.2f}%)"
)
print(
    f"[MODEL-FULL] exact fitted-support pixels={n_fitted_support}, continuation-only pixels={n_cont_only}"
)
print(
    f"[MODEL-FULL] resid_full finite={int(np.isfinite(resid_full).sum())}, "
    f"model_full finite={int(np.isfinite(model_full).sum())}"
)


## Clipping science residual

Вычитаем `model_full` из science image и ограничиваем экстремальные residual pixels для SBF-измерения. Цель: подавить сильные non-SBF выбросы, сохранив поле пиксельных флуктуаций.


In [ ]:
print("building sigma-capped full-frame science residual for SBF annuli...")

if "resid_full" not in globals() or "model_full" not in globals():
    raise RuntimeError("[RESID-CLIP] run the extrapolated full-frame model cell before sigma clipping")

science_clip_mask = (~premask) & np.isfinite(resid_full) & np.isfinite(model_full) & (model_full > 0.0)
n_science_clip = int(science_clip_mask.sum())
if n_science_clip < MIN_PIXELS_SBF:
    raise RuntimeError(f"[RESID-CLIP] too few valid science pixels: N={n_science_clip}")

vals_science_clip = resid_full[science_clip_mask]
mean_clip_full, med_clip_full, std_clip_full = sigma_clipped_stats(
    vals_science_clip,
    sigma=CLIP_SIGMA_QC,
    maxiters=CLIP_MAXIT_QC,
)

if (not np.isfinite(std_clip_full)) or std_clip_full <= 0.0:
    raise RuntimeError(f"[RESID-CLIP] bad clipped std={std_clip_full}")

clip_lo_full = float(med_clip_full - CLIP_SIGMA_QC * std_clip_full)
clip_hi_full = float(med_clip_full + CLIP_SIGMA_QC * std_clip_full)

resid_full_clip = np.array(resid_full, dtype=np.float32, copy=True)
resid_full_clip[science_clip_mask] = np.clip(
    resid_full_clip[science_clip_mask],
    clip_lo_full,
    clip_hi_full,
)
resid_full_clip[~science_clip_mask] = np.nan

resid_full_clip_delta = np.full_like(resid_full_clip, np.nan, dtype=np.float32)
resid_full_clip_delta[science_clip_mask] = (
    resid_full_clip[science_clip_mask] - resid_full[science_clip_mask]
)

n_clip_changed = int(np.count_nonzero(np.abs(resid_full_clip_delta[science_clip_mask]) > 0.0))
clip_tag = CLIP_TAG_QC

print(
    f"[RESID-CLIP] sigma={CLIP_SIGMA_QC}, valid={n_science_clip}, "
    f"med={med_clip_full:.3e}, std={std_clip_full:.3e}"
)
print(
    f"[RESID-CLIP] cap range = [{clip_lo_full:.3e}, {clip_hi_full:.3e}], "
    f"changed={n_clip_changed} px ({100.0 * n_clip_changed / n_science_clip:.2f}%)"
)


In [ ]:
print("saving sigma-capped science residual FITS right after clipping...")

resid_science_clip_path = out_dir / f"{stem}_sbf_resid_full_science_clip_{CLIP_TAG_QC}sigma.fits"
resid_science_delta_path = out_dir / f"{stem}_sbf_resid_full_science_clip_{CLIP_TAG_QC}sigma_delta.fits"
resid_science_path = out_dir / f"{stem}_sbf_resid_full_science.fits"

fits.writeto(resid_science_clip_path, np.array(resid_full_clip, dtype=np.float32), hdr150, overwrite=True)
fits.writeto(resid_science_delta_path, np.array(resid_full_clip_delta, dtype=np.float32), hdr150, overwrite=True)
fits.writeto(resid_science_path, np.array(resid_full_clip, dtype=np.float32), hdr150, overwrite=True)

print(f"[OUT] science residual clipped  -> {resid_science_clip_path}")
print(f"[OUT] clip delta                -> {resid_science_delta_path}")
print(f"[OUT] science residual used     -> {resid_science_path}")


In [ ]:
print("saving raw and sigma-capped full-frame residual FITS...")

model_for_view = model_full if "model_full" in globals() else model
model_support = np.isfinite(model_for_view) & (model_for_view > 0.0)
model_zero = np.where(model_support, model_for_view, 0.0)
resid_full_frame = np.array(img - model_zero, dtype=np.float32, copy=True)
resid_full_frame[premask | ~np.isfinite(resid_full_frame)] = np.nan

resid_full_frame_path = out_dir / f"{stem}_sbf_resid_full_frame.fits"
resid_full_masked_path = out_dir / f"{stem}_sbf_resid_full_masked.fits"
model_support_path = out_dir / f"{stem}_sbf_model_support_mask.fits"
model_full_path = out_dir / f"{stem}_sbf_model_full.fits"
resid_science_raw_path = out_dir / f"{stem}_sbf_resid_full_science_raw.fits"
resid_science_clip_path = out_dir / f"{stem}_sbf_resid_full_science_clip_{CLIP_TAG_QC}sigma.fits"
resid_science_delta_path = out_dir / f"{stem}_sbf_resid_full_science_clip_{CLIP_TAG_QC}sigma_delta.fits"
resid_science_path = out_dir / f"{stem}_sbf_resid_full_science.fits"

hdr_resid_full = hdr150.copy()
hdr_resid_full["BUNIT"] = hdr150.get("BUNIT", "")
hdr_resid_full.add_history("Full-frame residual view for inspection")
hdr_resid_full.add_history("Where model is finite: background-subtracted SCI - smooth model")
hdr_resid_full.add_history("Where model is not finite: background-subtracted SCI is kept unchanged")
hdr_resid_full.add_history("Pixels in premask or non-finite output are stored as NaN")

fits.writeto(resid_full_frame_path, resid_full_frame, hdr_resid_full, overwrite=True)
fits.writeto(resid_full_masked_path, resid_full_frame, hdr_resid_full, overwrite=True)
fits.writeto(model_support_path, model_support.astype(np.uint8), hdr150, overwrite=True)
fits.writeto(model_full_path, np.array(model_for_view, dtype=np.float32), hdr150, overwrite=True)

if "resid_full" in globals():
    fits.writeto(resid_science_raw_path, np.array(resid_full, dtype=np.float32), hdr150, overwrite=True)

if "resid_full_clip" in globals():
    fits.writeto(resid_science_clip_path, np.array(resid_full_clip, dtype=np.float32), hdr150, overwrite=True)
    fits.writeto(resid_science_delta_path, np.array(resid_full_clip_delta, dtype=np.float32), hdr150, overwrite=True)
    fits.writeto(resid_science_path, np.array(resid_full_clip, dtype=np.float32), hdr150, overwrite=True)
elif "resid_full" in globals():
    fits.writeto(resid_science_path, np.array(resid_full, dtype=np.float32), hdr150, overwrite=True)

finite_full = np.isfinite(resid_full_frame)
print(f"[OUT] full-frame residual view -> {resid_full_frame_path}")
print(f"[OUT] compatibility copy      -> {resid_full_masked_path}")
print(f"[OUT] model support mask       -> {model_support_path}")
print(f"[OUT] smooth model             -> {model_full_path}")
if "resid_full" in globals():
    print(f"[OUT] science residual raw      -> {resid_science_raw_path}")
if "resid_full_clip" in globals():
    print(f"[OUT] science residual clipped  -> {resid_science_clip_path}")
    print(f"[OUT] clip delta                -> {resid_science_delta_path}")
print(f"[OUT] science residual used     -> {resid_science_path}")
print(
    f"[RESID-FULL] finite={int(finite_full.sum())} px, "
    f"model_support={int(model_support.sum())} px "
    f"({100.0 * model_support.sum() / model_support.size:.2f}%)"
)
print(
    f"[RESID-FULL] med={np.nanmedian(resid_full_frame):.3e}, "
    f"std={np.nanstd(resid_full_frame):.3e}"
)


## Единый science residual path

Выбираем единую пару residual/model, которую обязаны использовать все дальнейшие SBF-измерения. Это защищает от случайного использования устаревших промежуточных residual.


In [ ]:
print("selecting unified science residual/model path for all spectral SBF measurements...")

if "model_full" not in globals() or "resid_full_clip" not in globals():
    raise RuntimeError("[SCIENCE-PATH] run the full-frame model and sigma-capped residual cells before spectral SBF")

science_model = model_full
science_resid = resid_full_clip
science_model_name = "model_full"
science_resid_name = f"resid_full_clip_{CLIP_TAG_QC}sigma"

if science_model.shape != science_resid.shape:
    raise RuntimeError(
        f"[SCIENCE-PATH] shape mismatch: model={science_model.shape}, resid={science_resid.shape}"
    )

def _science_array_stats(arr):
    good = np.isfinite(arr)
    if not np.any(good):
        return {
            "shape": tuple(arr.shape),
            "n_finite": 0,
            "min": np.nan,
            "median": np.nan,
            "max": np.nan,
            "std": np.nan,
        }
    vals = arr[good]
    return {
        "shape": tuple(arr.shape),
        "n_finite": int(good.sum()),
        "min": float(np.nanmin(vals)),
        "median": float(np.nanmedian(vals)),
        "max": float(np.nanmax(vals)),
        "std": float(np.nanstd(vals)),
    }

science_model_stats = _science_array_stats(science_model)
science_resid_stats = _science_array_stats(science_resid)

print(f"[SCIENCE-PATH] science_model = {science_model_name}")
print(f"[SCIENCE-PATH] science_resid = {science_resid_name}")
print(
    f"[SCIENCE-PATH][model] shape={science_model_stats['shape']}, finite={science_model_stats['n_finite']}, "
    f"min={science_model_stats['min']:.3e}, med={science_model_stats['median']:.3e}, "
    f"max={science_model_stats['max']:.3e}, std={science_model_stats['std']:.3e}"
)
print(
    f"[SCIENCE-PATH][resid] shape={science_resid_stats['shape']}, finite={science_resid_stats['n_finite']}, "
    f"min={science_resid_stats['min']:.3e}, med={science_resid_stats['median']:.3e}, "
    f"max={science_resid_stats['max']:.3e}, std={science_resid_stats['std']:.3e}"
)


## Дополнительная residual-mask

Строим дополнительную compact-source mask уже на model-subtracted residual. Она целится в яркие остатки шаровиков и других compact sources, которые пережили первичную маску и sigma clipping.


In [ ]:
print("building supplemental compact residual mask from current science residual...")

if "science_resid" not in globals() or "science_model" not in globals():
    raise RuntimeError("[RESID-MASK] unified science residual/model are unavailable; rerun the science-path cell")
if "img" not in globals() or "model_full" not in globals():
    raise RuntimeError("[RESID-MASK] img/model_full are unavailable; rerun the model cell")
if "premask" not in globals():
    raise RuntimeError("[RESID-MASK] premask is unavailable; rerun the premask cell")

RESID_EXTRA_MASK_SIGMA = 3
RESID_EXTRA_MASK_NPIX = 9
RESID_EXTRA_MASK_KERNEL_SIGMA = 1.0
RESID_EXTRA_MASK_DILATE = 2
RESID_EXTRA_MASK_MAX_AREA = 3000
RESID_EXTRA_MASK_POSITIVE_ONLY = True
RESID_EXTRA_MASK_BRANCH = "compactresidmask"

if "premask_base" not in globals():
    premask_base = np.array(premask, dtype=bool, copy=True)
if "premask_src_base" not in globals() and "premask_src" in globals():
    premask_src_base = np.array(premask_src, dtype=bool, copy=True)

detect_support = (~premask_base) & np.isfinite(science_resid) & np.isfinite(science_model) & (science_model > 0.0)
n_detect_support = int(detect_support.sum())
if n_detect_support < MIN_PIXELS_SBF:
    raise RuntimeError(f"[RESID-MASK] too few science pixels for residual-mask detection: N={n_detect_support}")

vals_det = np.asarray(science_resid[detect_support], dtype=float)
_, med_det, std_det = sigma_clipped_stats(vals_det, sigma=CLIP_SIGMA_QC, maxiters=CLIP_MAXIT_QC)
if (not np.isfinite(std_det)) or std_det <= 0.0:
    raise RuntimeError(f"[RESID-MASK] bad residual std for compact-mask detection: std={std_det}")

resid_det = np.zeros(science_resid.shape, dtype=np.float32)
resid_centered = np.asarray(science_resid - med_det, dtype=np.float32)
if RESID_EXTRA_MASK_POSITIVE_ONLY:
    resid_det[detect_support] = np.maximum(resid_centered[detect_support], 0.0)
else:
    resid_det[detect_support] = np.abs(resid_centered[detect_support])

if RESID_EXTRA_MASK_KERNEL_SIGMA > 0.0:
    resid_det_smooth = gaussian_filter(resid_det, sigma=RESID_EXTRA_MASK_KERNEL_SIGMA, mode="nearest")
else:
    resid_det_smooth = resid_det

threshold_det = float(RESID_EXTRA_MASK_SIGMA * std_det)
segm_extra = detect_sources(resid_det_smooth, threshold_det, npixels=RESID_EXTRA_MASK_NPIX)

extra_resid_mask_raw = np.zeros_like(premask_base, dtype=bool)
extra_resid_mask = np.zeros_like(premask_base, dtype=bool)
df_resid_extra_sources = pd.DataFrame(columns=["label", "area_pix", "peak_resid", "peak_snr"])

if segm_extra is not None:
    seg_data = np.asarray(segm_extra.data, dtype=int)
    labels = np.unique(seg_data)
    labels = labels[labels > 0]
    if labels.size > 0:
        areas_all = np.bincount(seg_data.ravel())
        peak_vals = ndimage.maximum(resid_det, labels=seg_data, index=labels)
        peak_snr = peak_vals / std_det
        keep_rows = []
        keep_labels = []
        for lab, peak_val, peak_sig in zip(labels, peak_vals, peak_snr):
            area_pix = int(areas_all[int(lab)])
            if area_pix > RESID_EXTRA_MASK_MAX_AREA:
                continue
            if not np.isfinite(peak_val) or peak_val <= threshold_det:
                continue
            keep_labels.append(int(lab))
            keep_rows.append({
                "label": int(lab),
                "area_pix": area_pix,
                "peak_resid": float(peak_val),
                "peak_snr": float(peak_sig),
            })
        if keep_labels:
            extra_resid_mask_raw = np.isin(seg_data, np.asarray(keep_labels, dtype=int)) & detect_support
            if RESID_EXTRA_MASK_DILATE > 0:
                extra_resid_mask = ndimage.binary_dilation(extra_resid_mask_raw, iterations=RESID_EXTRA_MASK_DILATE)
            else:
                extra_resid_mask = extra_resid_mask_raw.copy()
            extra_resid_mask &= detect_support
            df_resid_extra_sources = pd.DataFrame(keep_rows).sort_values(["peak_snr", "area_pix"], ascending=[False, False]).reset_index(drop=True)

premask = np.array(premask_base | extra_resid_mask, dtype=bool)
premask_extra_resid = np.array(extra_resid_mask, dtype=bool, copy=True)

resid_full = np.array(img - model_full, dtype=np.float32, copy=True)
resid_full[premask | (~np.isfinite(model_full)) | (~np.isfinite(img))] = np.nan

science_clip_mask = (~premask) & np.isfinite(resid_full) & np.isfinite(model_full) & (model_full > 0.0)
n_science_clip = int(science_clip_mask.sum())
if n_science_clip < MIN_PIXELS_SBF:
    raise RuntimeError(f"[RESID-MASK] too few valid science pixels after extra mask: N={n_science_clip}")

vals_science_clip = np.asarray(resid_full[science_clip_mask], dtype=float)
mean_clip_full, med_clip_full, std_clip_full = sigma_clipped_stats(
    vals_science_clip,
    sigma=CLIP_SIGMA_QC,
    maxiters=CLIP_MAXIT_QC,
)
if (not np.isfinite(std_clip_full)) or std_clip_full <= 0.0:
    raise RuntimeError(f"[RESID-MASK] bad clipped std after extra mask: std={std_clip_full}")

clip_lo_full = float(med_clip_full - CLIP_SIGMA_QC * std_clip_full)
clip_hi_full = float(med_clip_full + CLIP_SIGMA_QC * std_clip_full)
resid_full_clip = np.array(resid_full, dtype=np.float32, copy=True)
resid_full_clip[science_clip_mask] = np.clip(resid_full_clip[science_clip_mask], clip_lo_full, clip_hi_full)
resid_full_clip[~science_clip_mask] = np.nan

resid_full_clip_delta = np.full_like(resid_full_clip, np.nan, dtype=np.float32)
resid_full_clip_delta[science_clip_mask] = resid_full_clip[science_clip_mask] - resid_full[science_clip_mask]

science_model = model_full
science_resid = resid_full_clip
science_model_name = "model_full"
science_resid_name = f"resid_full_clip_{CLIP_TAG_QC}sigma_plus_{RESID_EXTRA_MASK_BRANCH}"

n_raw_extra = int(extra_resid_mask_raw.sum())
n_final_extra = int(extra_resid_mask.sum())
n_clip_changed = int(np.count_nonzero(np.abs(resid_full_clip_delta[science_clip_mask]) > 0.0))

extra_mask_path = out_dir / f"{stem}_sbf_resid_extra_mask_{RESID_EXTRA_MASK_BRANCH}.fits"
premask_path = out_dir / f"{stem}_sbf_premask_{RESID_EXTRA_MASK_BRANCH}.fits"
resid_science_clip_path_branch = out_dir / f"{stem}_sbf_resid_full_science_clip_{CLIP_TAG_QC}sigma_{RESID_EXTRA_MASK_BRANCH}.fits"
resid_science_delta_path_branch = out_dir / f"{stem}_sbf_resid_full_science_clip_{CLIP_TAG_QC}sigma_delta_{RESID_EXTRA_MASK_BRANCH}.fits"

fits.writeto(extra_mask_path, np.asarray(extra_resid_mask, dtype=np.uint8), hdr150, overwrite=True)
fits.writeto(premask_path, np.asarray(premask, dtype=np.uint8), hdr150, overwrite=True)
fits.writeto(resid_science_clip_path_branch, np.asarray(resid_full_clip, dtype=np.float32), hdr150, overwrite=True)
fits.writeto(resid_science_delta_path_branch, np.asarray(resid_full_clip_delta, dtype=np.float32), hdr150, overwrite=True)

print(
    f"[RESID-MASK] residual sigma for compact leftover detection: med={med_det:.3e}, std={std_det:.3e}, "
    f"threshold={threshold_det:.3e} ({RESID_EXTRA_MASK_SIGMA:.1f} sigma)"
)
print(
    f"[RESID-MASK] kept compact residual labels={len(df_resid_extra_sources)}, raw_mask={n_raw_extra} px, "
    f"dilated_mask={n_final_extra} px, total premask coverage={100.0 * premask.sum() / premask.size:.2f}%"
)
print(
    f"[RESID-MASK] updated science clip: valid={n_science_clip}, med={med_clip_full:.3e}, std={std_clip_full:.3e}, "
    f"cap=[{clip_lo_full:.3e}, {clip_hi_full:.3e}], changed={n_clip_changed} px ({100.0 * n_clip_changed / n_science_clip:.2f}%)"
)
print(f"[OUT] extra residual mask      -> {extra_mask_path}")
print(f"[OUT] combined premask        -> {premask_path}")
print(f"[OUT] branch science residual -> {resid_science_clip_path_branch}")
print(f"[OUT] branch clip delta       -> {resid_science_delta_path_branch}")
print("[RESID-MASK] rerun the spectral SBF cells below to propagate the updated premask/science_resid.")
display(df_resid_extra_sources.head(20))


## Каталог compact sources для $P_r$

Создаём каталог компактных источников для оценки residual variance от unresolved или частично замаскированных contaminant sources. Этот каталог нужен для коррекции `P_r`, а не для построения гладкой модели галактики.


In [ ]:
print("building compact-source catalog for unresolved-source variance correction...")

jy_per_pix = MJY_SR_TO_JY_PER_ARCSEC2 * pix_area
source_catalog_ref_mask = np.isfinite(model_full) & (model_full > 0.0)
source_catalog_columns = [
    "label", "xcentroid", "ycentroid", "area_pix",
    "flux_img_signed", "flux_img_positive", "flux_img_adopted",
    "flux_jy", "mag_ab",
]

if not np.any(source_catalog_ref_mask):
    raise RuntimeError("[SRC-CAT] model_full has no positive pixels for reference mask")

source_catalog_seg_full = np.zeros(model_full.shape, dtype=np.int32)
source_catalog_build_info = {}
compact_source_catalog = None

yy_ref, xx_ref = np.where(source_catalog_ref_mask)
y0_src, y1_src = int(yy_ref.min()), int(yy_ref.max()) + 1
x0_src, x1_src = int(xx_ref.min()), int(xx_ref.max()) + 1
source_catalog_bbox = (y0_src, y1_src, x0_src, x1_src)

ref_mask_crop = np.asarray(source_catalog_ref_mask[y0_src:y1_src, x0_src:x1_src], dtype=bool)
catalog_image = np.array(
    img[y0_src:y1_src, x0_src:x1_src] - model_full[y0_src:y1_src, x0_src:x1_src],
    dtype=float,
    copy=True,
)
catalog_image[~np.isfinite(catalog_image)] = np.nan
catalog_image[~ref_mask_crop] = np.nan

valid_detect = ref_mask_crop & np.isfinite(catalog_image)
detect_values = catalog_image[valid_detect]
if detect_values.size < MIN_PIXELS_SIGMA_CLIP:
    raise RuntimeError(f"[SRC-CAT] too few valid residual pixels in model-support crop: N={detect_values.size}")

mean_det, med_det, std_det = sigma_clipped_stats(
    detect_values,
    sigma=SIGMA_STAT,
    maxiters=SIGMA_MAXIT,
)
threshold_det = float(med_det + SBF_PR_DET_SIGMA * max(std_det, ROBUST_SCALE_FLOOR))

detect_image = np.zeros_like(catalog_image, dtype=float)
detect_image[valid_detect] = catalog_image[valid_detect]

segm_detect = detect_sources(
    detect_image,
    threshold=threshold_det,
    npixels=SBF_PR_DET_NPIXELS,
    connectivity=SOURCE_DETECT_CONNECTIVITY,
)
n_detect_raw = int(segm_detect.nlabels) if segm_detect is not None else 0

if segm_detect is not None and SBF_PR_DO_DEBLEND:
    try:
        segm_detect = deblend_sources(
            detect_image,
            segm_detect,
            npixels=SBF_PR_DET_NPIXELS,
            nlevels=DEBLEND_NLEVELS,
            contrast=DEBLEND_CONTRAST,
            nproc=DEBLEND_NPROC,
            progress_bar=False,
        )
    except Exception as err:
        print(f"[SRC-CAT] deblend skipped: {err}")

n_detect_deblend = int(segm_detect.nlabels) if segm_detect is not None else 0

premask_overlap_count = np.nan
if "premask_segm" in globals() and premask_segm is not None and "premask_compact_labels" in globals():
    premask_crop = np.asarray(premask_segm.data[y0_src:y1_src, x0_src:x1_src], dtype=int)
    premask_overlap_labels = np.intersect1d(
        np.unique(premask_crop[ref_mask_crop & (premask_crop > 0)]),
        np.asarray(premask_compact_labels, dtype=int),
        assume_unique=False,
    )
    premask_overlap_count = int(premask_overlap_labels.size)

source_catalog_df = pd.DataFrame(columns=source_catalog_columns)
source_catalog_seg_crop = np.zeros_like(detect_image, dtype=np.int32)
dropped_large_area = 0
dropped_nonpositive_signed_flux = 0
dropped_nonfinite_mag = 0
n_compact_area = 0
n_final = 0
source_pixels_final = 0

if segm_detect is not None and segm_detect.nlabels > 0:
    seg_data = np.asarray(segm_detect.data, dtype=np.int32)
    seg_data[~ref_mask_crop] = 0

    labels_all = np.asarray(segm_detect.labels, dtype=int)
    max_label = int(seg_data.max()) if seg_data.size else 0
    counts = np.bincount(seg_data.ravel(), minlength=max_label + 1) if max_label > 0 else np.zeros(1, dtype=int)
    counts[0] = 0

    compact_labels = labels_all[(counts[labels_all] > 0) & (counts[labels_all] <= SBF_PR_MAX_COMPACT_AREA)]
    dropped_large_area = int(labels_all.size - compact_labels.size)
    n_compact_area = int(compact_labels.size)

    label_lut = np.zeros(max_label + 1, dtype=np.int32)
    if compact_labels.size > 0:
        label_lut[compact_labels] = compact_labels.astype(np.int32)
    source_catalog_seg_crop = label_lut[seg_data]

    ys_src, xs_src = np.nonzero(source_catalog_seg_crop > 0)
    if ys_src.size > 0:
        labels_src = source_catalog_seg_crop[ys_src, xs_src].astype(np.int64, copy=False)
        labels_present = np.unique(labels_src)
        values_src = np.nan_to_num(catalog_image[ys_src, xs_src], nan=0.0)
        positive_values_src = np.clip(values_src, 0.0, None)

        signed_sum = np.bincount(labels_src, weights=values_src, minlength=max_label + 1)
        positive_sum = np.bincount(labels_src, weights=positive_values_src, minlength=max_label + 1)
        area_sum = np.bincount(labels_src, minlength=max_label + 1)
        x_sum = np.bincount(labels_src, weights=xs_src.astype(float), minlength=max_label + 1)
        y_sum = np.bincount(labels_src, weights=ys_src.astype(float), minlength=max_label + 1)

        rows = []
        kept_labels = []
        for label in labels_present.astype(int):
            area_pix = int(area_sum[label])
            if area_pix <= 0:
                continue

            flux_img_signed = float(signed_sum[label])
            flux_img_positive = float(positive_sum[label])
            flux_img_adopted = flux_img_signed

            if not (np.isfinite(flux_img_adopted) and flux_img_adopted > 0.0):
                dropped_nonpositive_signed_flux += 1
                continue

            flux_jy = float(flux_img_adopted * jy_per_pix)
            mag_ab = float(-2.5 * np.log10(flux_jy / AB_ZEROPOINT_JY)) if flux_jy > 0.0 else np.nan
            if not np.isfinite(mag_ab):
                dropped_nonfinite_mag += 1
                continue

            kept_labels.append(label)
            rows.append({
                "label": label,
                "xcentroid": float(x0_src + x_sum[label] / area_pix),
                "ycentroid": float(y0_src + y_sum[label] / area_pix),
                "area_pix": area_pix,
                "flux_img_signed": flux_img_signed,
                "flux_img_positive": flux_img_positive,
                "flux_img_adopted": flux_img_adopted,
                "flux_jy": flux_jy,
                "mag_ab": mag_ab,
            })

        source_catalog_df = pd.DataFrame(rows, columns=source_catalog_columns)
        if not source_catalog_df.empty:
            source_catalog_df = source_catalog_df.sort_values(["mag_ab", "label"], na_position="last").reset_index(drop=True)

        final_lut = np.zeros(max_label + 1, dtype=np.int32)
        if kept_labels:
            final_lut[np.asarray(kept_labels, dtype=int)] = np.asarray(kept_labels, dtype=np.int32)
        source_catalog_seg_crop = final_lut[source_catalog_seg_crop]
        source_catalog_seg_full[y0_src:y1_src, x0_src:x1_src] = source_catalog_seg_crop
        n_final = int(len(kept_labels))
        source_pixels_final = int(np.count_nonzero(source_catalog_seg_crop))

source_catalog_build_info.update({
    "bbox": source_catalog_bbox,
    "support_pixels": int(source_catalog_ref_mask.sum()),
    "n_detect_raw": int(n_detect_raw),
    "n_detect_deblend": int(n_detect_deblend),
    "n_compact_area": int(n_compact_area),
    "dropped_large_area": int(dropped_large_area),
    "dropped_nonpositive_signed_flux": int(dropped_nonpositive_signed_flux),
    "dropped_nonfinite_mag": int(dropped_nonfinite_mag),
    "n_final": int(n_final),
    "source_pixels_final": int(source_pixels_final),
    "premask_overlap_count": premask_overlap_count,
    "detect_threshold": float(threshold_det),
    "detect_med": float(med_det),
    "detect_std": float(std_det),
    "detect_image": SBF_PR_SOURCE_IMAGE,
})

source_catalog_csv_path = out_dir / f"{stem}_sbf_compact_source_catalog.csv"
source_catalog_df.to_csv(source_catalog_csv_path, index=False)

finite_mag = np.isfinite(source_catalog_df.get("mag_ab", pd.Series(dtype=float))).sum() if not source_catalog_df.empty else 0
print(
    f"[SRC-CAT] detect image={SBF_PR_SOURCE_IMAGE}, crop={y0_src}:{y1_src}, {x0_src}:{x1_src}, "
    f"support pixels={int(source_catalog_ref_mask.sum())}"
)
print(
    f"[SRC-CAT] residual stats in support: med={med_det:.3e}, std={std_det:.3e}, "
    f"threshold={threshold_det:.3e}"
)
if np.isfinite(premask_overlap_count):
    print(f"[SRC-CAT] old premask-overlap inside model support = {int(premask_overlap_count)} compact labels")
print(
    f"[SRC-CAT] segmentation: raw={n_detect_raw}, deblended={n_detect_deblend}, "
    f"compact-area pass={n_compact_area}, dropped_large={dropped_large_area}"
)
print(
    f"[SRC-CAT] photometry filter: kept={n_final}, dropped_nonpositive_signed_flux={dropped_nonpositive_signed_flux}, "
    f"dropped_nonfinite_mag={dropped_nonfinite_mag}, source pixels kept={source_pixels_final}"
)
print(
    f"[SRC-CAT] compact catalog size={len(source_catalog_df)}, finite mags={finite_mag}, "
    f"reference pixels={int(source_catalog_ref_mask.sum())}"
)
print(f"[OUT] compact source catalog -> {source_catalog_csv_path}")


## Helpers для unresolved-source variance

Определяем функции luminosity function и source-overlap, которые оценивают `P_r`. Итоговая SBF-амплитуда дальше считается как corrected fluctuation power `P_fluc = P0 - P_r`.


In [ ]:
print("preparing unresolved-source variance helpers...")

def sbf_ab_zeropoint_from_pix_area(pix_area):
    jy_per_pix_local = MJY_SR_TO_JY_PER_ARCSEC2 * pix_area
    return float(-2.5 * np.log10(jy_per_pix_local / AB_ZEROPOINT_JY))

def sbf_flux_image_units_from_mag(mag_ab, pix_area):
    zp_ab_local = sbf_ab_zeropoint_from_pix_area(pix_area)
    return float(10.0 ** (-0.4 * (float(mag_ab) - zp_ab_local)))

def optional_finite_float(value):
    try:
        value_f = float(value)
    except (TypeError, ValueError):
        return np.nan
    return value_f if np.isfinite(value_f) else np.nan

def select_sources_in_region_mask(region_mask, region_origin_x=0, region_origin_y=0):
    if source_catalog_df.empty:
        return source_catalog_df.copy()

    region_mask = np.asarray(region_mask, dtype=bool)
    ny_mask, nx_mask = region_mask.shape

    seg_full = globals().get("source_catalog_seg_full", None)
    if isinstance(seg_full, np.ndarray) and seg_full.ndim == 2:
        y0 = int(region_origin_y)
        x0 = int(region_origin_x)
        y1 = y0 + ny_mask
        x1 = x0 + nx_mask
        if 0 <= y0 < y1 <= seg_full.shape[0] and 0 <= x0 < x1 <= seg_full.shape[1]:
            seg_view = np.asarray(seg_full[y0:y1, x0:x1], dtype=int)
            labels_in_region = np.unique(seg_view[region_mask & (seg_view > 0)])
            if labels_in_region.size == 0:
                return source_catalog_df.iloc[0:0].copy()
            keep = source_catalog_df["label"].isin(labels_in_region.astype(int))
            return source_catalog_df.loc[keep].copy().reset_index(drop=True)

    keep = []
    for idx, row in source_catalog_df.iterrows():
        xcen = row.get("xcentroid", np.nan)
        ycen = row.get("ycentroid", np.nan)
        if not (np.isfinite(xcen) and np.isfinite(ycen)):
            continue

        ix = int(np.rint(float(xcen))) - int(region_origin_x)
        iy = int(np.rint(float(ycen))) - int(region_origin_y)
        if 0 <= ix < nx_mask and 0 <= iy < ny_mask and bool(region_mask[iy, ix]):
            keep.append(idx)

    if not keep:
        return source_catalog_df.iloc[0:0].copy()
    return source_catalog_df.loc[keep].copy().reset_index(drop=True)

def summarize_sources_in_region_mask(region_mask, region_origin_x=0, region_origin_y=0):
    reg_sources = select_sources_in_region_mask(
        region_mask,
        region_origin_x=region_origin_x,
        region_origin_y=region_origin_y,
    )
    if reg_sources.empty:
        return {
            "n_overlap": 0,
            "n_valid": 0,
            "n_nonpositive_flux": 0,
            "mag_min": np.nan,
            "mag_med": np.nan,
            "mag_max": np.nan,
        }

    flux = reg_sources["flux_jy"].to_numpy(dtype=float) if "flux_jy" in reg_sources else np.full(len(reg_sources), np.nan)
    mags = reg_sources["mag_ab"].to_numpy(dtype=float) if "mag_ab" in reg_sources else np.full(len(reg_sources), np.nan)
    valid = np.isfinite(flux) & (flux > SBF_PR_CATALOG_MIN_FLUX_JY) & np.isfinite(mags)
    mags_valid = mags[valid]

    return {
        "n_overlap": int(len(reg_sources)),
        "n_valid": int(np.count_nonzero(valid)),
        "n_nonpositive_flux": int(np.count_nonzero(~(np.isfinite(flux) & (flux > SBF_PR_CATALOG_MIN_FLUX_JY)))),
        "mag_min": float(np.nanmin(mags_valid)) if mags_valid.size > 0 else np.nan,
        "mag_med": float(np.nanmedian(mags_valid)) if mags_valid.size > 0 else np.nan,
        "mag_max": float(np.nanmax(mags_valid)) if mags_valid.size > 0 else np.nan,
    }

def estimate_turnover_magnitude(mags, mag_bin=None):
    if mag_bin is None:
        mag_bin = SBF_PR_MAG_BIN
    arr = np.asarray(mags, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size < 3:
        return np.nan, {"method": "too_few_sources", "n_sources": int(arr.size)}

    lo = float(np.floor(np.nanmin(arr) / mag_bin) * mag_bin)
    hi = float(np.ceil(np.nanmax(arr) / mag_bin) * mag_bin + mag_bin)
    edges = np.arange(lo, hi + 0.5 * mag_bin, mag_bin)
    if edges.size < 3:
        return np.nan, {"method": "bad_edges", "n_sources": int(arr.size)}

    counts, edges = np.histogram(arr, bins=edges)
    if not np.any(counts > 0):
        return np.nan, {"method": "empty_hist", "n_sources": int(arr.size)}

    idx_peak = int(np.argmax(counts))
    m_turn = float(0.5 * (edges[idx_peak] + edges[idx_peak + 1]))
    return m_turn, {
        "method": "turnover_hist",
        "n_sources": int(arr.size),
        "peak_count": int(counts[idx_peak]),
        "n_bins": int(counts.size),
    }

def fit_combined_contaminant_lf(mags, area_pix, m_lim, fit_width=None, mag_bin=None, gamma_fallback=None):
    if fit_width is None:
        fit_width = SBF_PR_FIT_WIDTH
    if mag_bin is None:
        mag_bin = SBF_PR_MAG_BIN
    if gamma_fallback is None or not np.isfinite(gamma_fallback):
        gamma_fallback = SBF_PR_DEFAULT_GAMMA

    arr = np.asarray(mags, dtype=float)
    arr = arr[np.isfinite(arr)]
    area_pix = float(area_pix)
    out = {
        "lf_model": "combined_power_law",
        "lf_gamma": np.nan,
        "lf_phi_mlim": np.nan,
        "lf_fit_method": "unavailable",
        "lf_fit_rmse": np.nan,
        "lf_n_fit_bins": 0,
        "lf_n_fit_sources": 0,
        "lf_fit_mbright": float(m_lim - fit_width),
        "lf_fit_mfaint": float(m_lim),
    }

    if arr.size == 0 or (not np.isfinite(m_lim)) or area_pix <= 0.0:
        out["lf_fit_method"] = "no_sources_or_bad_limit"
        return out

    m_lo = float(m_lim - fit_width)
    fit_sel = (arr >= m_lo) & (arr <= m_lim)
    fit_mags = arr[fit_sel]
    out["lf_n_fit_sources"] = int(fit_mags.size)

    edges = np.arange(m_lo, m_lim + mag_bin * 1.01, mag_bin)
    if edges.size < 3:
        out["lf_fit_method"] = "bad_edges"
        return out

    counts, edges = np.histogram(fit_mags, bins=edges)
    centers = 0.5 * (edges[:-1] + edges[1:])
    positive = counts > 0
    out["lf_n_fit_bins"] = int(np.count_nonzero(positive))

    gamma_min, gamma_max = map(float, SBF_PR_GAMMA_BOUNDS)
    local_fit_ok = out["lf_n_fit_bins"] >= int(SBF_PR_MIN_FIT_BINS)

    if local_fit_ok:
        x = centers[positive]
        y = np.log10(counts[positive] / (area_pix * mag_bin))
        w = np.sqrt(counts[positive].astype(float))
        try:
            coeff = np.polyfit(x, y, 1, w=w)
            gamma = float(coeff[0])
            intercept = float(coeff[1])
            if np.isfinite(gamma) and gamma_min <= gamma <= gamma_max:
                y_fit = intercept + gamma * x
                rmse = float(np.sqrt(np.nanmean((y - y_fit) ** 2))) if x.size > 0 else np.nan
                phi_mlim = float(10.0 ** (intercept + gamma * m_lim))
                if np.isfinite(phi_mlim) and phi_mlim > 0.0:
                    out.update({
                        "lf_gamma": gamma,
                        "lf_phi_mlim": phi_mlim,
                        "lf_fit_method": "local_histogram",
                        "lf_fit_rmse": rmse,
                    })
                    return out
        except Exception as err:
            out["lf_fit_method"] = f"local_histogram_failed: {err}"

    gamma = float(np.clip(gamma_fallback, gamma_min, gamma_max))
    span_integral = (1.0 - 10.0 ** (-gamma * fit_width)) / (gamma * np.log(10.0)) if gamma > 0.0 else float(fit_width)
    n_window = int(np.count_nonzero(fit_sel))
    if area_pix > 0.0 and span_integral > 0.0 and n_window > 0:
        phi_mlim = float((n_window / area_pix) / span_integral)
        out.update({
            "lf_gamma": gamma,
            "lf_phi_mlim": phi_mlim,
            "lf_fit_method": "local_window_norm_fixed_gamma",
        })
    else:
        out.update({
            "lf_gamma": gamma,
            "lf_fit_method": "fixed_gamma_no_local_norm",
        })
    return out

def integrate_pr_from_powerlaw_lf(phi_mlim, gamma, m_lim, pix_area):
    if not (np.isfinite(phi_mlim) and phi_mlim > 0.0 and np.isfinite(gamma) and np.isfinite(m_lim)):
        return np.nan
    if gamma >= 0.8:
        return np.nan

    f_lim_img = sbf_flux_image_units_from_mag(m_lim, pix_area)
    return float(phi_mlim * (f_lim_img ** 2) / ((0.8 - gamma) * np.log(10.0)))

def build_global_pr_reference():
    if source_catalog_df.empty:
        return {
            "region": "global_science_reference",
            "n_detected_sources": 0,
            "area_pix": int(source_catalog_ref_mask.sum()),
            "m_lim": np.nan,
            "m_lim_method": "empty_catalog",
            "lf_gamma": np.nan,
            "lf_phi_mlim": np.nan,
            "lf_fit_method": "empty_catalog",
            "Pr": np.nan,
        }

    ref_sources = select_sources_in_region_mask(source_catalog_ref_mask, region_origin_x=0, region_origin_y=0)
    ref_sources = ref_sources[
        np.isfinite(ref_sources.get("mag_ab", np.nan))
        & np.isfinite(ref_sources.get("flux_jy", np.nan))
        & (ref_sources.get("flux_jy", np.nan) > SBF_PR_CATALOG_MIN_FLUX_JY)
    ].copy()

    ref_mags = ref_sources["mag_ab"].to_numpy(dtype=float) if not ref_sources.empty else np.array([], dtype=float)
    m_lim_override = optional_finite_float(SBF_PR_MLIM_OVERRIDE)
    if np.isfinite(m_lim_override):
        m_lim = float(m_lim_override)
        m_lim_method = "manual_override"
    else:
        m_turn, turn_info = estimate_turnover_magnitude(ref_mags)
        if np.isfinite(m_turn) and ref_mags.size >= SBF_PR_MIN_SOURCES_GLOBAL:
            m_lim = float(m_turn + SBF_PR_MLIM_OFFSET)
            m_lim_method = f"global_{turn_info.get('method', 'turnover')}"
        elif ref_mags.size > 0:
            m_lim = float(np.nanquantile(ref_mags, SBF_PR_FALLBACK_QUANTILE) + SBF_PR_MLIM_OFFSET)
            m_lim_method = "global_quantile_fallback"
        else:
            m_lim = np.nan
            m_lim_method = "no_global_sources"

    fit = fit_combined_contaminant_lf(
        ref_mags,
        area_pix=float(source_catalog_ref_mask.sum()),
        m_lim=m_lim,
        gamma_fallback=SBF_PR_DEFAULT_GAMMA,
    )
    pr_value = integrate_pr_from_powerlaw_lf(fit.get("lf_phi_mlim", np.nan), fit.get("lf_gamma", np.nan), m_lim, pix_area)

    out = {
        "region": "global_science_reference",
        "n_detected_sources": int(ref_mags.size),
        "area_pix": int(source_catalog_ref_mask.sum()),
        "m_lim": m_lim,
        "m_lim_method": m_lim_method,
        "Pr": pr_value,
        **fit,
    }
    return out

def estimate_pr_for_region(region_mask, model, region_name="region", region_origin_x=0, region_origin_y=0):
    if not SBF_PR_ENABLE:
        return {
            "Pr": 0.0,
            "Pr_over_P0": np.nan,
            "n_detected_sources": 0,
            "n_detected_sources_total": 0,
            "m_lim": np.nan,
            "m_lim_method": "disabled",
            "lf_model": "disabled",
            "lf_gamma": np.nan,
            "lf_phi_mlim": np.nan,
            "lf_fit_method": "disabled",
            "lf_fit_rmse": np.nan,
            "lf_n_fit_bins": 0,
            "lf_n_fit_sources": 0,
            "lf_fit_mbright": np.nan,
            "lf_fit_mfaint": np.nan,
            "pr_note": "Pr correction disabled",
        }

    analysis_mask = np.asarray(region_mask, dtype=bool) & np.isfinite(model) & (model > 0.0)
    area_pix = int(np.count_nonzero(analysis_mask))
    reg_sources = select_sources_in_region_mask(analysis_mask, region_origin_x=region_origin_x, region_origin_y=region_origin_y)
    reg_sources_valid = reg_sources[
        np.isfinite(reg_sources.get("mag_ab", np.nan))
        & np.isfinite(reg_sources.get("flux_jy", np.nan))
        & (reg_sources.get("flux_jy", np.nan) > SBF_PR_CATALOG_MIN_FLUX_JY)
    ].copy()
    reg_mags = reg_sources_valid["mag_ab"].to_numpy(dtype=float) if not reg_sources_valid.empty else np.array([], dtype=float)

    global_ref = globals().get("sbf_pr_global_reference", None)
    global_gamma = global_ref.get("lf_gamma", np.nan) if isinstance(global_ref, dict) else np.nan
    global_phi = global_ref.get("lf_phi_mlim", np.nan) if isinstance(global_ref, dict) else np.nan
    global_mlim = global_ref.get("m_lim", np.nan) if isinstance(global_ref, dict) else np.nan

    m_lim_override = optional_finite_float(SBF_PR_MLIM_OVERRIDE)
    if np.isfinite(m_lim_override):
        m_lim = float(m_lim_override)
        m_lim_method = "manual_override"
    else:
        m_turn_local, local_turn_info = estimate_turnover_magnitude(reg_mags)
        if np.isfinite(m_turn_local) and reg_mags.size >= SBF_PR_MIN_SOURCES_REGION:
            m_lim = float(m_turn_local + SBF_PR_MLIM_OFFSET)
            m_lim_method = f"region_{local_turn_info.get('method', 'turnover')}"
        elif np.isfinite(global_mlim):
            m_lim = float(global_mlim)
            m_lim_method = "global_reference_turnover"
        elif reg_mags.size > 0:
            m_lim = float(np.nanquantile(reg_mags, SBF_PR_FALLBACK_QUANTILE) + SBF_PR_MLIM_OFFSET)
            m_lim_method = "region_quantile_fallback"
        else:
            m_lim = np.nan
            m_lim_method = "no_region_sources"

    fit = fit_combined_contaminant_lf(
        reg_mags,
        area_pix=float(area_pix),
        m_lim=m_lim,
        gamma_fallback=global_gamma,
    )

    pr_note = []
    if (not np.isfinite(fit.get("lf_phi_mlim", np.nan)) or fit.get("lf_phi_mlim", np.nan) <= 0.0) and np.isfinite(global_phi) and global_phi > 0.0:
        fit["lf_phi_mlim"] = float(global_phi)
        fit["lf_fit_method"] = str(fit.get("lf_fit_method", "unavailable")) + "+global_phi"
        pr_note.append("used global LF normalization")

    pr_value = integrate_pr_from_powerlaw_lf(fit.get("lf_phi_mlim", np.nan), fit.get("lf_gamma", np.nan), m_lim, pix_area)
    if np.isfinite(pr_value) and np.isfinite(global_ref.get("Pr", np.nan) if isinstance(global_ref, dict) else np.nan):
        pass
    elif np.isfinite(global_ref.get("Pr", np.nan) if isinstance(global_ref, dict) else np.nan):
        pr_value = float(global_ref["Pr"])
        pr_note.append("used global Pr fallback")

    if np.isfinite(pr_value) and np.isfinite(area_pix) and area_pix > 0 and reg_mags.size == 0:
        pr_note.append("no local detected sources; correction inherited from global reference")

    if np.isfinite(pr_value) and pr_value < 0.0:
        pr_value = np.nan
        pr_note.append("negative Pr clipped to NaN")

    if not pr_note:
        pr_note.append("local combined contaminant LF")

    return {
        "Pr": pr_value,
        "Pr_over_P0": np.nan,
        "n_detected_sources": int(reg_mags.size),
        "n_detected_sources_total": int(len(reg_sources)),
        "m_lim": m_lim,
        "m_lim_method": m_lim_method,
        "lf_model": fit.get("lf_model", "combined_power_law"),
        "lf_gamma": fit.get("lf_gamma", np.nan),
        "lf_phi_mlim": fit.get("lf_phi_mlim", np.nan),
        "lf_fit_method": fit.get("lf_fit_method", "unavailable"),
        "lf_fit_rmse": fit.get("lf_fit_rmse", np.nan),
        "lf_n_fit_bins": fit.get("lf_n_fit_bins", 0),
        "lf_n_fit_sources": fit.get("lf_n_fit_sources", 0),
        "lf_fit_mbright": fit.get("lf_fit_mbright", np.nan),
        "lf_fit_mfaint": fit.get("lf_fit_mfaint", np.nan),
        "pr_note": "; ".join(pr_note),
    }

def solve_sbf_power_budget(P0, Imean, Pr, pix_area, P0_fit_sigma=np.nan):
    out = {
        "measurement_ok": False,
        "failure_reason": "",
        "P0": float(P0) if np.isfinite(P0) else np.nan,
        "Pr": float(Pr) if np.isfinite(Pr) else np.nan,
        "P_fluc": np.nan,
        "Pr_over_P0": np.nan,
        "Imean": float(Imean) if np.isfinite(Imean) else np.nan,
        "Pf_spec_raw": np.nan,
        "Pf_spec": np.nan,
        "mbar_spec_raw": np.nan,
        "mbar_spec": np.nan,
        "Pf_spec_sigma_formal": np.nan,
        "Pf_spec_sigma_formal_raw": np.nan,
        "mbar_fit_sigma": np.nan,
        "mbar_fit_sigma_raw": np.nan,
    }

    if not (np.isfinite(P0) and P0 > 0.0):
        out["failure_reason"] = f"bad raw P0={P0}"
        return out
    if not (np.isfinite(Imean) and Imean > 0.0):
        out["failure_reason"] = f"bad Imean={Imean}"
        return out

    zp_ab_local = sbf_ab_zeropoint_from_pix_area(pix_area)
    Pf_spec_raw = float(P0 / Imean)
    out["Pf_spec_raw"] = Pf_spec_raw
    if np.isfinite(Pf_spec_raw) and Pf_spec_raw > 0.0:
        out["mbar_spec_raw"] = float(-2.5 * np.log10(Pf_spec_raw) + zp_ab_local)
        if np.isfinite(P0_fit_sigma) and P0_fit_sigma > 0.0:
            out["Pf_spec_sigma_formal_raw"] = float(Pf_spec_raw * P0_fit_sigma / P0)
            out["mbar_fit_sigma_raw"] = float((2.5 / np.log(10.0)) * P0_fit_sigma / P0)

    if not np.isfinite(Pr):
        out["failure_reason"] = "Pr is not finite"
        return out

    P_fluc = float(P0 - Pr)
    out["P_fluc"] = P_fluc
    out["Pr_over_P0"] = float(Pr / P0) if np.isfinite(Pr) else np.nan
    if not (np.isfinite(P_fluc) and P_fluc > 0.0):
        out["failure_reason"] = f"non-positive corrected power: P0={P0:.3e}, Pr={Pr:.3e}, P_fluc={P_fluc:.3e}"
        return out

    Pf_spec = float(P_fluc / Imean)
    out["Pf_spec"] = Pf_spec
    if not (np.isfinite(Pf_spec) and Pf_spec > 0.0):
        out["failure_reason"] = f"bad corrected Pf_spec={Pf_spec}"
        return out

    out["mbar_spec"] = float(-2.5 * np.log10(Pf_spec) + zp_ab_local)
    if np.isfinite(P0_fit_sigma) and P0_fit_sigma > 0.0:
        out["Pf_spec_sigma_formal"] = float(Pf_spec * P0_fit_sigma / P_fluc)
        out["mbar_fit_sigma"] = float((2.5 / np.log(10.0)) * P0_fit_sigma / P_fluc)

    out["measurement_ok"] = True
    return out

sbf_pr_global_reference = build_global_pr_reference()
print(
    f"[Pr-GLOBAL] Nsrc={sbf_pr_global_reference.get('n_detected_sources', 0)}, "
    f"m_lim={sbf_pr_global_reference.get('m_lim', np.nan):.3f}, "
    f"gamma={sbf_pr_global_reference.get('lf_gamma', np.nan):.3f}, "
    f"Pr={sbf_pr_global_reference.get('Pr', np.nan):.3e}, "
    f"fit={sbf_pr_global_reference.get('lf_fit_method', 'n/a')}"
)


## Сохранение модели и residual

Сохраняем FITS-продукты гладкой модели и residual для визуальной проверки. Это основные файлы, которые надо смотреть глазами после запуска чистого пайплайна.


In [ ]:
model_path = out_dir / f"{stem}_sbf_model.fits"
resid_path = out_dir / f"{stem}_sbf_resid.fits"

fits.writeto(model_path, model, hdr150, overwrite=True)
fits.writeto(resid_path, resid, hdr150, overwrite=True)

print(f"[OUT] model → {model_path}")
print(f"[OUT] resid → {resid_path}")


## Helper спектрального SBF-фита

Определяем FFT-based процедуру SBF-измерения. Она строит expectation spectrum из PSF и smooth model, фитит `P(k) = P0 E(k) + P1`, оценивает unresolved-source correction и возвращает raw/corrected fluctuation magnitudes.


## PSF для expectation spectrum

Строим detector-sampled PSF через локальный `stpsf` и локальный WSS OPD, без MAST-запросов и без кэша. Нормированная переменная `psf` нужна дальше для построения `E(k)` в спектральном SBF-фите.


In [29]:
print("loading/building PSF...")

import os

psf_file = PSFREF if PSFREF is not None else f150w_path
psf_cache_path = out_dir / f"{stem}_psf_{PSF_SIZE}.fits"
print(f"[PSF] cache disabled for this run; ignoring {psf_cache_path}")

stpsf_data_dir = Path.home() / "data" / "stpsf-data"
if not stpsf_data_dir.exists():
    raise RuntimeError(f"[PSF] local STPSF data dir not found: {stpsf_data_dir}")
os.environ["STPSF_PATH"] = str(stpsf_data_dir)
print(f"[PSF] STPSF_PATH={stpsf_data_dir}")

local_wss_dir = stpsf_data_dir / "MAST_JWST_WSS_OPDs"
local_wss_files = sorted(local_wss_dir.glob("*.fits"))
if not local_wss_files:
    raise RuntimeError(f"[PSF] no local WSS OPD files found in {local_wss_dir}")
if len(local_wss_files) > 1:
    print(f"[PSF][WARN] found {len(local_wss_files)} local WSS OPD files; using the newest by mtime")
local_wss_opd = max(local_wss_files, key=lambda p: p.stat().st_mtime)
print(f"[PSF] local WSS OPD = {local_wss_opd}")

science_hdr = fits.getheader(str(psf_file), 0)
print(
    f"[PSF] science meta: INSTRUME={science_hdr.get('INSTRUME')} DETECTOR={science_hdr.get('DETECTOR')} "
    f"FILTER={science_hdr.get('FILTER')} PUPIL={science_hdr.get('PUPIL')} APERNAME={science_hdr.get('APERNAME')}"
)
print(f"[PSF] science date: {science_hdr.get('DATE-OBS')}T{science_hdr.get('TIME-OBS')}")
opd_hdr = fits.getheader(str(local_wss_opd), 0)
print(
    f"[PSF] OPD meta: CORR_ID={opd_hdr.get('CORR_ID', 'NA')} DATE-OBS={opd_hdr.get('DATE-OBS', opd_hdr.get('DATE', 'NA'))} "
    f"CONTENTS={opd_hdr.get('CONTENTS', 'NA')}"
)

sim = stpsf.instrument(science_hdr['INSTRUME'])
if (sim.name == 'NIRCam') and (science_hdr['PUPIL'][0] == 'F') and (science_hdr['PUPIL'][-1] in ['N', 'M']):
    sim.filter = science_hdr['PUPIL']
else:
    sim.filter = science_hdr['FILTER']
sim.set_position_from_aperture_name(science_hdr['APERNAME'])
if (sim.name == 'NIRCam') and (science_hdr.get('PUPIL', 'CLEAR') != 'CLEAR') and (not science_hdr['PUPIL'].startswith('F')) and (not science_hdr['PUPIL'].startswith('MASK')):
    sim.pupil_mask = science_hdr['PUPIL']
print(
    f"[PSF] configured local sim: instrument={sim.name} filter={sim.filter} detector={sim.detector} "
    f"apername={sim.aperturename} det_pos={sim.detector_position}"
)

sim.load_wss_opd(str(local_wss_opd), verbose=True, plot=False)
sim.options['output_mode'] = 'both'
psf_hdul = sim.calc_psf(
    nlambda=PSF_NLAMBDA,
    fov_pixels=PSF_SIZE,
    fft_oversample=4,
    detector_oversample=1,
    add_distortion=True,
)

def _psf_array_from_hdu(hdu):
    arr = np.array(hdu.data, dtype=float)
    if arr.ndim == 3:
        arr = arr.sum(axis=0)
    return arr

psf_selected_ext = None
if isinstance(psf_hdul, fits.HDUList):
    ext_rows = []
    for i, hdu in enumerate(psf_hdul):
        if getattr(hdu, 'data', None) is None:
            continue
        arr = np.array(hdu.data)
        extname = hdu.header.get('EXTNAME', 'PRIMARY' if i == 0 else f'EXT{i}')
        ext_rows.append(
            (
                i,
                extname,
                arr.shape,
                hdu.header.get('OVERSAMP', 'NA'),
                hdu.header.get('DET_SAMP', 'NA'),
                hdu.header.get('PIXELSCL', 'NA'),
            )
        )
    print('[PSF] returned HDU summary:')
    for row in ext_rows:
        print(f"    idx={row[0]} ext={row[1]} shape={row[2]} OVERSAMP={row[3]} DET_SAMP={row[4]} PIXELSCL={row[5]}")

    for extname in ('DET_DIST', 'DET_SAMP'):
        try:
            psf = _psf_array_from_hdu(psf_hdul[extname])
            psf_selected_ext = extname
            break
        except Exception:
            pass
    else:
        detector_like = []
        for i, hdu in enumerate(psf_hdul):
            if getattr(hdu, 'data', None) is None:
                continue
            arr = _psf_array_from_hdu(hdu)
            if arr.shape == (PSF_SIZE, PSF_SIZE):
                detector_like.append((hdu.header.get('EXTNAME', f'EXT{i}'), arr))
        if detector_like:
            psf_selected_ext, psf = detector_like[0]
        else:
            psf_selected_ext = 'PRIMARY_FALLBACK'
            psf = _psf_array_from_hdu(psf_hdul[0])
elif isinstance(psf_hdul, fits.PrimaryHDU):
    psf_selected_ext = 'PRIMARY'
    psf = _psf_array_from_hdu(psf_hdul)
else:
    psf_selected_ext = 'ARRAY'
    psf = np.array(psf_hdul, dtype=float)

print(f"[PSF] selected extension = {psf_selected_ext}")
psf = np.nan_to_num(psf, nan=0.0)
psf_sum = float(psf.sum())
if (not np.isfinite(psf_sum)) or (psf_sum <= 0.0):
    raise RuntimeError(f"[PSF] invalid PSF sum={psf_sum} for shape={psf.shape}")
psf /= psf_sum

if psf.shape != (PSF_SIZE, PSF_SIZE):
    print(
        f"[PSF][WARN] detector-scale selection still gave shape={psf.shape} for requested fov_pixels={PSF_SIZE}. "
        "E(k) may still be using the wrong sampling."
    )
print(f"[PSF] final shape={psf.shape}, sum={psf.sum():.6f}, min={psf.min():.3e}, max={psf.max():.3e}")


[11:23:53] loading/building PSF...
[11:23:53] [PSF] cache disabled for this run; ignoring data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_psf_129.fits
[11:23:53] [PSF] STPSF_PATH=/Users/zuha/data/stpsf-data
[11:23:53] [PSF] local WSS OPD = /Users/zuha/data/stpsf-data/MAST_JWST_WSS_OPDs/R2023111002-NRCA3_FP1-1.fits
[11:23:53] [PSF] science meta: INSTRUME=NIRCAM DETECTOR=MULTIPLE FILTER=F150W PUPIL=CLEAR APERNAME=NRCA1_FULL
[11:23:53] [PSF] science date: 2023-11-09T21:14:47.240
[11:23:53] [PSF] OPD meta: CORR_ID=R2023111002 DATE-OBS=2023-11-10 CONTENTS=NA
[11:23:54] [PSF] configured local sim: instrument=NIRCam filter=F150W detector=NRCA1 apername=NRCA1_FULL det_pos=(1024, 1024)
Importing and format-converting OPD from /Users/zuha/data/stpsf-data/MAST_JWST_WSS_OPDs/R2023111002-NRCA3_FP1-1.fits
Backing out SI WFE and OTE field dependence at the WF sensing field point (NRCA3_FP1)
[11:23:58] [PSF] returned HDU summary:
[11:23:58]     idx=0 ext=OVERSAMP shape=(129, 129) OVERSAMP=4 DET

In [30]:
def measure_sbf_spectral_region(
    region_mask,
    resid,
    model,
    premask,
    psf,
    pix_area,
    kmin=None,
    kmax=None,
    fft_workers=None,
    n_e_realizations=None,
    kbins_n=None,
    region_name="region",
    region_role="science",
    region_origin_x=0,
    region_origin_y=0,
):
    if kmin is None or kmax is None:
        kmin, kmax = FFT_K_RANGE_MAIN
    if fft_workers is None:
        fft_workers = FFT_WORKERS
    if n_e_realizations is None:
        n_e_realizations = FFT_E_REALIZATIONS_DIAG
    if kbins_n is None:
        kbins_n = FFT_KBINS_N

    mask_sbf_local = premask | (~region_mask)
    window = (~mask_sbf_local) & np.isfinite(resid) & np.isfinite(model) & (model > 0.0)

    n_use = int(window.sum())
    if n_use < MIN_PIXELS_SBF:
        return None

    Imean = float(np.nanmean(model[window]))
    if (not np.isfinite(Imean)) or (Imean <= 0.0):
        return None

    data_fft = np.zeros_like(resid, dtype=float)
    data_fft[window] = resid[window]
    data_fft[window] -= float(np.nanmean(data_fft[window]))

    with set_workers(fft_workers):
        F = fft2(data_fft)

    P2d = (np.abs(F) ** 2) / float(n_use)

    ny, nx = resid.shape
    ky = fftshift(fftfreq(ny))
    kx = fftshift(fftfreq(nx))
    KX, KY = np.meshgrid(kx, ky)
    kr = np.hypot(KX, KY)

    P2d_s = fftshift(P2d)
    kbins = np.linspace(0.0, float(kr.max()), kbins_n)

    Pk_vals = np.full(len(kbins) - 1, np.nan, dtype=float)
    Pk_k = np.full_like(Pk_vals, np.nan)
    for i in range(len(Pk_vals)):
        sel = (kr >= kbins[i]) & (kr < kbins[i + 1])
        vals = P2d_s[sel]
        if vals.size >= MIN_POINTS_PK_BIN:
            Pk_vals[i] = float(np.nanmedian(vals))
            Pk_k[i] = 0.5 * (kbins[i] + kbins[i + 1])

    mP = np.isfinite(Pk_vals) & np.isfinite(Pk_k) & (Pk_k > 0.0)
    if int(mP.sum()) < MIN_POINTS_FIT:
        return None

    kP = Pk_k[mP]
    P = Pk_vals[mP]

    big_psf = np.zeros((ny, nx), dtype=float)
    py, px = psf.shape
    y0_psf = ny // 2 - py // 2
    x0_psf = nx // 2 - px // 2
    big_psf[y0_psf:y0_psf + py, x0_psf:x0_psf + px] = psf

    with set_workers(fft_workers):
        F_psf = fft2(big_psf)

    rng = np.random.default_rng(FFT_RNG_SEED)
    Ek_stack = []
    for _ in range(n_e_realizations):
        noise = rng.normal(loc=0.0, scale=1.0, size=(ny, nx))
        with set_workers(fft_workers):
            F_noise = fft2(noise)
            sim = np.real(np.fft.ifft2(F_noise * F_psf))

        sim_masked = np.zeros_like(sim, dtype=float)
        sim_masked[window] = sim[window]
        sim_masked[window] -= float(np.nanmean(sim_masked[window]))

        with set_workers(fft_workers):
            F_sim = fft2(sim_masked)

        E2d_sim = (np.abs(F_sim) ** 2) / float(n_use)
        E2d_sim_s = fftshift(E2d_sim)

        Ek_vals_i = np.full(len(kbins) - 1, np.nan, dtype=float)
        for j in range(len(Ek_vals_i)):
            sel = (kr >= kbins[j]) & (kr < kbins[j + 1])
            vals = E2d_sim_s[sel]
            if vals.size >= MIN_POINTS_PK_BIN:
                Ek_vals_i[j] = float(np.nanmedian(vals))
        Ek_stack.append(Ek_vals_i)

    Ek_stack = np.array(Ek_stack, dtype=float)
    Ek_vals = np.nanmedian(Ek_stack, axis=0)
    Ek_k = 0.5 * (kbins[:-1] + kbins[1:])

    mE = np.isfinite(Ek_vals) & np.isfinite(Ek_k) & (Ek_k > 0.0)
    if int(mE.sum()) < MIN_POINTS_FIT:
        return None

    kE = Ek_k[mE]
    E = Ek_vals[mE]

    sel = (kP >= kmin) & (kP <= kmax)
    if int(sel.sum()) < MIN_POINTS_FIT:
        return None

    E_int = np.interp(kP[sel], kE, E, left=np.nan, right=np.nan)
    x = E_int
    y = P[sel]
    good = np.isfinite(x) & np.isfinite(y) & (x > 0.0) & (y > 0.0)
    x = x[good]
    y = y[good]
    if x.size < MIN_POINTS_FIT:
        return None

    corr = float(np.corrcoef(x, y)[0, 1])
    A = np.vstack([x, np.ones_like(x)]).T
    P0, P1 = np.linalg.lstsq(A, y, rcond=None)[0]
    if (not np.isfinite(P0)) or (P0 <= 0.0):
        return None

    y_fit = P0 * x + P1
    fit_resid = y - y_fit
    fit_resid_std = float(np.nanstd(fit_resid, ddof=1)) if x.size > 2 else float(np.nanstd(fit_resid))

    P0_fit_sigma = np.nan
    mbar_fit_sigma_raw = np.nan
    dof = max(int(x.size) - 2, 1)
    try:
        sigma2_fit = float(np.nansum(fit_resid * fit_resid) / dof)
        cov_fit = sigma2_fit * np.linalg.pinv(A.T @ A)
        P0_fit_sigma = float(np.sqrt(max(cov_fit[0, 0], 0.0)))
        if np.isfinite(P0_fit_sigma) and P0_fit_sigma > 0.0:
            mbar_fit_sigma_raw = float((2.5 / np.log(10.0)) * P0_fit_sigma / P0)
    except Exception:
        pass

    den = P0 + P1
    frac = float(P0 / den) if np.isfinite(den) and den != 0.0 else np.nan

    pr_info = estimate_pr_for_region(
        region_mask=region_mask,
        model=model,
        region_name=region_name,
        region_origin_x=region_origin_x,
        region_origin_y=region_origin_y,
    )
    power_info = solve_sbf_power_budget(P0=P0, Imean=Imean, Pr=pr_info.get("Pr", np.nan), pix_area=pix_area, P0_fit_sigma=P0_fit_sigma)

    return {
        "measurement_ok": bool(power_info.get("measurement_ok", False)),
        "failure_reason": power_info.get("failure_reason", ""),
        "region_name": region_name,
        "region_role": region_role,
        "n_use": n_use,
        "Imean": Imean,
        "P0": float(P0),
        "P1": float(P1),
        "P_fluc": power_info.get("P_fluc", np.nan),
        "Pr": pr_info.get("Pr", np.nan),
        "Pr_over_P0": power_info.get("Pr_over_P0", np.nan),
        "frac": frac,
        "corr": corr,
        "n_fit": int(x.size),
        "fit_resid_std": fit_resid_std,
        "P0_fit_sigma": P0_fit_sigma,
        "mbar_fit_sigma_raw": mbar_fit_sigma_raw,
        "mbar_fit_sigma": power_info.get("mbar_fit_sigma", np.nan),
        "Pf_spec_raw": power_info.get("Pf_spec_raw", np.nan),
        "Pf_spec": power_info.get("Pf_spec", np.nan),
        "Pf_spec_sigma_formal_raw": power_info.get("Pf_spec_sigma_formal_raw", np.nan),
        "Pf_spec_sigma_formal": power_info.get("Pf_spec_sigma_formal", np.nan),
        "mbar_spec_raw": power_info.get("mbar_spec_raw", np.nan),
        "mbar_spec": power_info.get("mbar_spec", np.nan),
        "n_detected_sources": pr_info.get("n_detected_sources", 0),
        "n_detected_sources_total": pr_info.get("n_detected_sources_total", 0),
        "m_lim": pr_info.get("m_lim", np.nan),
        "m_lim_method": pr_info.get("m_lim_method", "unknown"),
        "lf_model": pr_info.get("lf_model", "combined_power_law"),
        "lf_gamma": pr_info.get("lf_gamma", np.nan),
        "lf_phi_mlim": pr_info.get("lf_phi_mlim", np.nan),
        "lf_fit_method": pr_info.get("lf_fit_method", "unknown"),
        "lf_fit_rmse": pr_info.get("lf_fit_rmse", np.nan),
        "lf_n_fit_bins": pr_info.get("lf_n_fit_bins", 0),
        "lf_n_fit_sources": pr_info.get("lf_n_fit_sources", 0),
        "lf_fit_mbright": pr_info.get("lf_fit_mbright", np.nan),
        "lf_fit_mfaint": pr_info.get("lf_fit_mfaint", np.nan),
        "pr_note": pr_info.get("pr_note", ""),
    }

def measure_sbf_spec_for_row(
    row,
    resid,
    model,
    premask,
    psf,
    pix_area,
    kmin=None,
    kmax=None,
    fft_workers=None,
    n_e_realizations=None,
    kbins_n=None,
):
    """
    Считает spectral SBF для одного isophote-window row.

    Важно: raw P0 сам по себе ещё не является stellar SBF signal, потому что
    unresolved contaminants тоже дают PSF-shaped power. Поэтому итогом считаем
    corrected power P_fluc = P0 - Pr.
    """
    ny_r, nx_r = resid.shape
    yy_r, xx_r = np.indices((ny_r, nx_r), dtype=float)

    sma_in = float(row["sma_in"])
    sma_out = float(row["sma_out"])
    x0_ann = float(row["x0"])
    y0_ann = float(row["y0"])
    pa_ann = float(row["pa"])
    q_ann = max(GEOM_Q_FLOOR, float(row["q"]))

    dx = xx_r - x0_ann
    dy = yy_r - y0_ann
    cosp = np.cos(pa_ann)
    sinp = np.sin(pa_ann)
    xp = dx * cosp + dy * sinp
    yp = -dx * sinp + dy * cosp
    r_ell = np.sqrt(xp * xp + (yp / q_ann) * (yp / q_ann))
    annulus_ell = (r_ell >= sma_in) & (r_ell <= sma_out)

    out = measure_sbf_spectral_region(
        region_mask=annulus_ell,
        resid=resid,
        model=model,
        premask=premask,
        psf=psf,
        pix_area=pix_area,
        kmin=kmin,
        kmax=kmax,
        fft_workers=fft_workers,
        n_e_realizations=n_e_realizations,
        kbins_n=kbins_n,
        region_name=f"plateau_{sma_in:.1f}_{sma_out:.1f}",
        region_role="diagnostic",
        region_origin_x=0,
        region_origin_y=0,
    )
    if out is None:
        return None

    out.update({
        "sma_in": sma_in,
        "sma_out": sma_out,
        "sma_mid": float(row["sma_mid"]),
        "i0": int(row["i0"]),
        "i1": int(row["i1"]),
    })
    return out


## SBF в круговых annuli

Определяем маски круговых annuli и запускаем спектральный SBF-фит в двух принятых областях: 8.2-16.4 arcsec и 16.4-32.8 arcsec. Основное science window задаётся `FFT_K_RANGE_MAIN`; остальные окна оставлены только как лёгкая проверка устойчивости.


In [31]:
def build_region_mask_ellipse(shape, x0, y0, q, pa, sma_in, sma_out):
    ny, nx = shape
    yy, xx = np.ogrid[:ny, :nx]

    dx = xx.astype(float) - float(x0)
    dy = yy.astype(float) - float(y0)

    cosp = np.cos(pa)
    sinp = np.sin(pa)

    xp = dx * cosp + dy * sinp
    yp = -dx * sinp + dy * cosp

    r_ell = np.sqrt(xp * xp + (yp / q) * (yp / q))
    return (r_ell >= sma_in) & (r_ell <= sma_out)

def build_region_mask_circle(shape, x0, y0, r_in, r_out):
    ny, nx = shape
    yy, xx = np.ogrid[:ny, :nx]
    rr2 = (xx.astype(float) - float(x0)) ** 2 + (yy.astype(float) - float(y0)) ** 2
    return (rr2 >= float(r_in) ** 2) & (rr2 <= float(r_out) ** 2)


In [32]:
def measure_sbf_for_mask(
    region_mask,
    resid,
    model,
    premask,
    psf,
    pix_area,
    kmin,
    kmax,
    fft_workers=None,
    n_e_realizations=None,
    kbins_n=None,
    region_name="region",
    region_role="science",
    region_origin_x=0,
    region_origin_y=0,
):
    return measure_sbf_spectral_region(
        region_mask=region_mask,
        resid=resid,
        model=model,
        premask=premask,
        psf=psf,
        pix_area=pix_area,
        kmin=kmin,
        kmax=kmax,
        fft_workers=fft_workers,
        n_e_realizations=n_e_realizations,
        kbins_n=kbins_n,
        region_name=region_name,
        region_role=region_role,
        region_origin_x=region_origin_x,
        region_origin_y=region_origin_y,
    )


In [33]:
print("measuring SBF in Jensen/TRGB-SBF circular annuli...")

if "science_model" not in globals() or "science_resid" not in globals():
    raise RuntimeError("[SBF-REGION] unified science path is unavailable; rerun the science-path cell")

pix_scale = float(np.sqrt(pix_area))
r1_in, r1_out = SBF_LIT_INNER_ARCSEC[0] / pix_scale, SBF_LIT_INNER_ARCSEC[1] / pix_scale
r2_in, r2_out = SBF_LIT_OUTER_ARCSEC[0] / pix_scale, SBF_LIT_OUTER_ARCSEC[1] / pix_scale

print(
    f"[SBF-REGION] unified science path: residual={science_resid_name}, model={science_model_name}; "
    "fixed circular annuli are the science regions"
)
print(
    f"[SBF-REGION] circular center = ({x0_sbf_circ:.2f}, {y0_sbf_circ:.2f}), "
    f"radii = {r1_in:.1f}-{r1_out:.1f} px and {r2_in:.1f}-{r2_out:.1f} px"
)

regions = [
    {
        "region": "circular_inner_lit",
        "role": "science",
        "shape": "circle",
        "mask": build_region_mask_circle(
            science_resid.shape, x0_sbf_circ, y0_sbf_circ, r1_in, r1_out
        ),
        "rin_px": r1_in,
        "rout_px": r1_out,
        "rin_arcsec": SBF_LIT_INNER_ARCSEC[0],
        "rout_arcsec": SBF_LIT_INNER_ARCSEC[1],
        "resid_data": science_resid,
        "model_data": science_model,
        "resid_source": science_resid_name,
        "model_source": science_model_name,
        "origin_x": 0,
        "origin_y": 0,
    },
    {
        "region": "circular_outer_lit",
        "role": "science",
        "shape": "circle",
        "mask": build_region_mask_circle(
            science_resid.shape, x0_sbf_circ, y0_sbf_circ, r2_in, r2_out
        ),
        "rin_px": r2_in,
        "rout_px": r2_out,
        "rin_arcsec": SBF_LIT_OUTER_ARCSEC[0],
        "rout_arcsec": SBF_LIT_OUTER_ARCSEC[1],
        "resid_data": science_resid,
        "model_data": science_model,
        "resid_source": science_resid_name,
        "model_source": science_model_name,
        "origin_x": 0,
        "origin_y": 0,
    },
]

k_windows = list(SBF_REGION_K_WINDOWS)
rows_df = []

for reg in regions:
    region_mask = reg["mask"]
    reg_resid = reg["resid_data"]
    reg_model = reg["model_data"]
    region_pixels = int(region_mask.sum())
    usable_region = region_mask & (~premask) & np.isfinite(reg_resid) & np.isfinite(reg_model) & (reg_model > 0.0)
    usable_pixels = int(usable_region.sum())
    usable_fraction = float(usable_pixels / region_pixels) if region_pixels > 0 else np.nan

    print(
        f"[SBF-REGION] {reg['region']} ({reg['role']}): "
        f"N_region={region_pixels}, N_usable={usable_pixels}, "
        f"usable_fraction={usable_fraction:.3f}, residual={reg['resid_source']}, model={reg['model_source']}"
    )
    src_reg_summary = summarize_sources_in_region_mask(
        region_mask,
        region_origin_x=reg['origin_x'],
        region_origin_y=reg['origin_y'],
    )
    print(
        f"[SRC-CAT-REGION] {reg['region']}: overlap={src_reg_summary['n_overlap']}, "
        f"valid={src_reg_summary['n_valid']}, nonpositive_flux={src_reg_summary['n_nonpositive_flux']}, "
        f"mag_med={src_reg_summary['mag_med']:.3f}"
    )

    for kmin, kmax in k_windows:
        out = measure_sbf_for_mask(
            region_mask=region_mask,
            resid=reg_resid,
            model=reg_model,
            premask=premask,
            psf=psf,
            pix_area=pix_area,
            kmin=kmin,
            kmax=kmax,
            fft_workers=FFT_WORKERS,
            n_e_realizations=FFT_E_REALIZATIONS_MAIN,
            kbins_n=FFT_KBINS_N,
            region_name=reg["region"],
            region_role=reg["role"],
            region_origin_x=reg["origin_x"],
            region_origin_y=reg["origin_y"],
        )

        base_row = {
            "region": reg["region"],
            "role": reg["role"],
            "shape": reg["shape"],
            "rin_px": reg["rin_px"],
            "rout_px": reg["rout_px"],
            "rin_arcsec": reg["rin_arcsec"],
            "rout_arcsec": reg["rout_arcsec"],
            "resid_source": reg["resid_source"],
            "model_source": reg["model_source"],
            "region_pixels": region_pixels,
            "usable_pixels": usable_pixels,
            "usable_fraction": usable_fraction,
            "kmin": kmin,
            "kmax": kmax,
        }

        if out is None:
            rows_df.append({**base_row, "status": "failed_raw"})
            print(f"[SBF-REGION]   k={kmin:.3f}..{kmax:.3f} -> raw spectral fit failed")
            continue

        status = "ok" if out.get("measurement_ok", False) else "invalid_corrected"
        rows_df.append({**base_row, "status": status, **out})

        if status == "ok":
            corr_text = f"{out['Pr_over_P0']:.3%}" if np.isfinite(out.get("Pr_over_P0", np.nan)) else "n/a"
            print(
                f"[SBF-REGION]   k={kmin:.3f}..{kmax:.3f}: raw mbar={out['mbar_spec_raw']:.4f}, "
                f"corrected mbar={out['mbar_spec']:.4f}, P0={out['P0']:.3e}, Pr={out['Pr']:.3e}, "
                f"P_fluc={out['P_fluc']:.3e}, Pr/P0={corr_text}, "
                f"Nsrc_valid={out['n_detected_sources']}, Nsrc_overlap={out.get('n_detected_sources_total', 0)}, "
                f"m_lim={out['m_lim']:.3f}, gamma={out['lf_gamma']:.3f}"
            )
        else:
            print(
                f"[SBF-REGION]   k={kmin:.3f}..{kmax:.3f}: raw mbar={out['mbar_spec_raw']:.4f}, "
                f"corrected measurement invalid ({out.get('failure_reason', '')}), "
                f"P0={out['P0']:.3e}, Pr={out['Pr']:.3e}, P_fluc={out['P_fluc']:.3e}"
            )

df_sbf = pd.DataFrame(rows_df)
display(df_sbf)


[11:23:58] measuring SBF in Jensen/TRGB-SBF circular annuli...
[11:23:58] [SBF-REGION] unified science path: residual=resid_full_clip_3p5sigma_plus_compactresidmask, model=model_full; fixed circular annuli are the science regions
[11:23:58] [SBF-REGION] circular center = (6607.66, 1168.96), radii = 262.6-525.2 px and 525.2-1050.4 px
[11:23:59] [SBF-REGION] circular_inner_lit (science): N_region=649924, N_usable=649088, usable_fraction=0.999, residual=resid_full_clip_3p5sigma_plus_compactresidmask, model=model_full
[11:23:59] [SRC-CAT-REGION] circular_inner_lit: overlap=4108, valid=4108, nonpositive_flux=0, mag_med=25.293
[11:33:11] [SBF-REGION]   k=0.010..0.250: raw mbar=27.8274, corrected mbar=27.8404, P0=1.715e+01, Pr=2.041e-01, P_fluc=1.694e+01, Pr/P0=1.190%, Nsrc_valid=4108, Nsrc_overlap=4108, m_lim=25.875, gamma=0.278
[11:42:20] [SBF-REGION]   k=0.030..0.250: raw mbar=27.9198, corrected mbar=27.9339, P0=1.575e+01, Pr=2.041e-01, P_fluc=1.555e+01, Pr/P0=1.296%, Nsrc_valid=4108, Nsrc

,region,role,shape,rin_px,rout_px,rin_arcsec,rout_arcsec,resid_source,model_source,region_pixels,...,lf_model,lf_gamma,lf_phi_mlim,lf_fit_method,lf_fit_rmse,lf_n_fit_bins,lf_n_fit_sources,lf_fit_mbright,lf_fit_mfaint,pr_note
0,circular_inner_lit,science,circle,262.597443,525.194886,8.2,16.4,resid_full_clip_3p5sigma_plus_compactresidmask,model_full,649924,...,combined_power_law,0.277958,0.004899,local_histogram,0.028310,6,3061,24.375,25.875,local combined contaminant LF
1,circular_inner_lit,science,circle,262.597443,525.194886,8.2,16.4,resid_full_clip_3p5sigma_plus_compactresidmask,model_full,649924,...,combined_power_law,0.277958,0.004899,local_histogram,0.028310,6,3061,24.375,25.875,local combined contaminant LF
2,circular_inner_lit,science,circle,262.597443,525.194886,8.2,16.4,resid_full_clip_3p5sigma_plus_compactresidmask,model_full,649924,...,combined_power_law,0.277958,0.004899,local_histogram,0.028310,6,3061,24.375,25.875,local combined contaminant LF
3,circular_outer_lit,science,circle,525.194886,1050.389772,16.4,32.8,resid_full_clip_3p5sigma_plus_compactresidmask,model_full,2599650,...,combined_power_law,0.627664,0.001824,local_histogram,0.035028,6,2283,24.375,25.875,local combined contaminant LF
4,circular_outer_lit,science,circle,525.194886,1050.389772,16.4,32.8,resid_full_clip_3p5sigma_plus_compactresidmask,model_full,2599650,...,combined_power_law,0.627664,0.001824,local_histogram,0.035028,6,2283,24.375,25.875,local combined contaminant LF
5,circular_outer_lit,science,circle,525.194886,1050.389772,16.4,32.8,resid_full_clip_3p5sigma_plus_compactresidmask,model_full,2599650,...,combined_power_law,0.627664,0.001824,local_histogram,0.035028,6,2283,24.375,25.875,local combined contaminant LF


## FITS с реально измеряемыми пикселями

Сохраняем два FITS-кадра, где оставлены только пиксели, реально входящие в SBF-измерение inner и outer annuli. Всё вне usable region записывается как `NaN`, чтобы было легко глазами проверить измеряемые данные.


In [34]:
print("saving usable science residual FITS for circular annuli...")

if "science_resid" not in globals() or "science_model" not in globals():
    raise RuntimeError("[SCIENCE-RESID-SAVE] unified science residual/model are unavailable; rerun the science-path cell")
if "premask" not in globals():
    raise RuntimeError("[SCIENCE-RESID-SAVE] premask is unavailable; rerun the premask cell")
if "x0_sbf_circ" not in globals() or "y0_sbf_circ" not in globals() or "pix_area" not in globals():
    raise RuntimeError("[SCIENCE-RESID-SAVE] circular geometry is unavailable; rerun the circular-annulus measurement cell")

pix_scale = float(np.sqrt(pix_area))
region_specs = [
    ("circular_inner_lit", SBF_LIT_INNER_ARCSEC[0] / pix_scale, SBF_LIT_INNER_ARCSEC[1] / pix_scale),
    ("circular_outer_lit", SBF_LIT_OUTER_ARCSEC[0] / pix_scale, SBF_LIT_OUTER_ARCSEC[1] / pix_scale),
]

for reg_name, rin_px, rout_px in region_specs:
    region_mask = build_region_mask_circle(science_resid.shape, x0_sbf_circ, y0_sbf_circ, rin_px, rout_px)
    usable_region = (
        region_mask
        & (~premask)
        & np.isfinite(science_resid)
        & np.isfinite(science_model)
        & (science_model > 0.0)
    )

    resid_use = np.full(science_resid.shape, np.nan, dtype=np.float32)
    resid_use[usable_region] = np.asarray(science_resid[usable_region], dtype=np.float32)

    fits_path = out_dir / f"{stem}_sbf_resid_science_{reg_name}_usable.fits"
    fits.writeto(fits_path, resid_use, hdr150, overwrite=True)

    print(
        f"[SCIENCE-RESID-SAVE] {reg_name}: residual={science_resid_name}, model={science_model_name}, "
        f"rin={rin_px:.1f}px, rout={rout_px:.1f}px, N_region={int(region_mask.sum())}, "
        f"N_usable={int(usable_region.sum())}, saved -> {fits_path}"
    )


[12:20:59] saving usable science residual FITS for circular annuli...
[12:21:00] [SCIENCE-RESID-SAVE] circular_inner_lit: residual=resid_full_clip_3p5sigma_plus_compactresidmask, model=model_full, rin=262.6px, rout=525.2px, N_region=649924, N_usable=649088, saved -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_science_circular_inner_lit_usable.fits
[12:21:00] [SCIENCE-RESID-SAVE] circular_outer_lit: residual=resid_full_clip_3p5sigma_plus_compactresidmask, model=model_full, rin=525.2px, rout=1050.4px, N_region=2599650, N_usable=2054404, saved -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_science_circular_outer_lit_usable.fits


## Raw и corrected SBF

Показываем таблицу по annuli: raw `P0`, коррекцию `P_r`, corrected fluctuation power и соответствующие magnitudes.


In [35]:
compare_columns = [
    "region", "role", "status", "kmin", "kmax",
    "mbar_spec_raw", "mbar_spec",
    "P0", "Pr", "P_fluc", "Pr_over_P0",
    "Imean", "n_detected_sources", "m_lim", "lf_gamma", "lf_fit_method",
]

compare_columns = [col for col in compare_columns if col in df_sbf.columns]
df_sbf_compare = df_sbf[compare_columns].copy()
display(df_sbf_compare.sort_values(["region", "kmin", "kmax"]).reset_index(drop=True))


,region,role,status,kmin,kmax,mbar_spec_raw,mbar_spec,P0,Pr,P_fluc,Pr_over_P0,Imean,n_detected_sources,m_lim,lf_gamma,lf_fit_method
0,circular_inner_lit,science,ok,0.01,0.25,27.827420,27.840419,17.148660,0.204095,16.944565,0.011901,14.634180,4108,25.875,0.277958,local_histogram
1,circular_inner_lit,science,ok,0.03,0.25,27.919762,27.933923,15.750466,0.204095,15.546371,0.012958,14.634180,4108,25.875,0.277958,local_histogram
2,circular_inner_lit,science,ok,0.04,0.25,27.944310,27.958796,15.398357,0.204095,15.194262,0.013254,14.634180,4108,25.875,0.277958,local_histogram
3,circular_outer_lit,science,ok,0.01,0.25,27.634459,27.668027,7.559565,0.230142,7.329423,0.030444,5.400708,3285,25.875,0.627664,local_histogram
4,circular_outer_lit,science,ok,0.03,0.25,27.752252,27.789733,6.782343,0.230142,6.552201,0.033933,5.400708,3285,25.875,0.627664,local_histogram
5,circular_outer_lit,science,ok,0.04,0.25,27.776376,27.814715,6.633308,0.230142,6.403165,0.034695,5.400708,3285,25.875,0.627664,local_histogram


## Итоговый SBF по фиксированным круговым annuli

Объединяем два fixed circular annuli в science-like result. Если оба annuli успешно измерены, предпочитается configured main k-window; итоговая ошибка берётся как максимум из formal weighted error и scatter между inner/outer annuli.


In [36]:
print("combining fixed circular SBF annuli with weighted mean (main science result)...")

required_annuli = ["circular_inner_lit", "circular_outer_lit"]
df_sbf_ok = df_sbf[df_sbf["status"].eq("ok")].copy()

def _finite_positive(value):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return False
    return np.isfinite(value) and value > 0.0

def _robust_mag_scatter(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]

    if arr.size >= 3:
        med = float(np.nanmedian(arr))
        mad_sigma = float(1.4826 * np.nanmedian(np.abs(arr - med)))
        std_sigma = float(np.nanstd(arr, ddof=1))
        if _finite_positive(mad_sigma):
            return mad_sigma, "k-window MAD"
        if _finite_positive(std_sigma):
            return std_sigma, "k-window std"

    if arr.size >= 2:
        std_sigma = float(np.nanstd(arr, ddof=1))
        half_range = float(0.5 * (np.nanmax(arr) - np.nanmin(arr)))
        if _finite_positive(std_sigma):
            return std_sigma, "k-window std"
        if _finite_positive(half_range):
            return half_range, "k-window half-range"

    return np.nan, "no k-window scatter"

region_sigma_info = {}
for region_name, grp in df_sbf_ok.groupby("region"):
    sigma_k, sigma_method = _robust_mag_scatter(grp["mbar_spec"].values)
    region_sigma_info[region_name] = {
        "sigma_kwindow": sigma_k,
        "sigma_method": sigma_method,
    }

def _fit_resid_proxy_mag(row):
    fit_resid_std = row.get("fit_resid_std", np.nan)
    power_ref = row.get("P_fluc", np.nan)
    if not _finite_positive(power_ref):
        power_ref = row.get("P0", np.nan)
    if _finite_positive(fit_resid_std) and _finite_positive(abs(power_ref)):
        return float((2.5 / np.log(10.0)) * fit_resid_std / abs(power_ref))
    return np.nan

def _measurement_sigma_for_row(row):
    candidates = []
    info = region_sigma_info.get(row["region"], {})

    sigma_k = info.get("sigma_kwindow", np.nan)
    if _finite_positive(sigma_k):
        candidates.append((float(sigma_k), info.get("sigma_method", "k-window scatter")))

    sigma_fit = row.get("mbar_fit_sigma", np.nan)
    if _finite_positive(sigma_fit):
        candidates.append((float(sigma_fit), "fit covariance"))

    if candidates:
        sigma, method = max(candidates, key=lambda item: item[0])
        if len(candidates) > 1:
            method = "max(" + ", ".join(m for _, m in candidates) + ")"
        return sigma, method

    sigma_proxy = _fit_resid_proxy_mag(row)
    if _finite_positive(sigma_proxy):
        return sigma_proxy, "fit_resid_std/P_fluc proxy"

    return np.nan, "sigma unavailable"

def _row_for(region_name, kmin_value, kmax_value):
    if df_sbf_ok.empty:
        return None
    match = df_sbf_ok[
        df_sbf_ok["region"].eq(region_name)
        & np.isclose(df_sbf_ok["kmin"].astype(float), float(kmin_value))
        & np.isclose(df_sbf_ok["kmax"].astype(float), float(kmax_value))
    ]
    if match.empty:
        return None
    return match.iloc[0]

def _annulus_measurement(region_name, kmin_value, kmax_value):
    row = _row_for(region_name, kmin_value, kmax_value)
    if row is None:
        return np.nan, np.nan, "not ok", None

    mbar = float(row["mbar_spec"]) if np.isfinite(row.get("mbar_spec", np.nan)) else np.nan
    sigma, sigma_method = _measurement_sigma_for_row(row)
    return mbar, sigma, sigma_method, row

pairs_from_df = df_sbf[df_sbf["region"].isin(required_annuli)][["kmin", "kmax"]].dropna().drop_duplicates()
k_pairs = sorted({(float(r.kmin), float(r.kmax)) for r in pairs_from_df.itertuples(index=False)})

sigma_pipeline = globals().get("sigma_pipeline", np.nan)
summary_rows = []
for kmin_value, kmax_value in k_pairs:
    mbar_inner, sigma_inner, method_inner, row_inner = _annulus_measurement(
        "circular_inner_lit", kmin_value, kmax_value
    )
    mbar_outer, sigma_outer, method_outer, row_outer = _annulus_measurement(
        "circular_outer_lit", kmin_value, kmax_value
    )

    mbar_inner_raw = float(row_inner["mbar_spec_raw"]) if row_inner is not None and np.isfinite(row_inner.get("mbar_spec_raw", np.nan)) else np.nan
    mbar_outer_raw = float(row_outer["mbar_spec_raw"]) if row_outer is not None and np.isfinite(row_outer.get("mbar_spec_raw", np.nan)) else np.nan
    inner_ok = np.isfinite(mbar_inner) and _finite_positive(sigma_inner)
    outer_ok = np.isfinite(mbar_outer) and _finite_positive(sigma_outer)
    both_measured = np.isfinite(mbar_inner) and np.isfinite(mbar_outer)

    mbar_weighted = np.nan
    mbar_weighted_raw = np.nan
    sigma_weighted_formal = np.nan
    annulus_scatter = float(abs(mbar_inner - mbar_outer) / 2.0) if both_measured else np.nan
    sigma_adopted = np.nan

    if inner_ok and outer_ok:
        mvals = np.array([mbar_inner, mbar_outer], dtype=float)
        sigmas = np.array([sigma_inner, sigma_outer], dtype=float)
        weights = 1.0 / (sigmas * sigmas)
        mbar_weighted = float(np.sum(weights * mvals) / np.sum(weights))
        sigma_weighted_formal = float(np.sqrt(1.0 / np.sum(weights)))
        if np.isfinite(mbar_inner_raw) and np.isfinite(mbar_outer_raw):
            mbar_weighted_raw = float(np.sum(weights * np.array([mbar_inner_raw, mbar_outer_raw])) / np.sum(weights))

        if _finite_positive(sigma_pipeline):
            sigma_adopted = float(
                np.sqrt(sigma_weighted_formal**2 + annulus_scatter**2 + float(sigma_pipeline)**2)
            )
            notes = "inner+outer weighted; adopted includes pipeline systematic"
        else:
            sigma_adopted = float(max(sigma_weighted_formal, annulus_scatter))
            notes = "inner+outer weighted; no pipeline systematic yet"

    elif inner_ok or outer_ok:
        if inner_ok:
            mbar_weighted = float(mbar_inner)
            mbar_weighted_raw = float(mbar_inner_raw) if np.isfinite(mbar_inner_raw) else np.nan
            sigma_weighted_formal = float(sigma_inner)
            sigma_adopted = float(sigma_inner)
            notes = "only circular_inner_lit ok; annulus scatter unavailable"
        else:
            mbar_weighted = float(mbar_outer)
            mbar_weighted_raw = float(mbar_outer_raw) if np.isfinite(mbar_outer_raw) else np.nan
            sigma_weighted_formal = float(sigma_outer)
            sigma_adopted = float(sigma_outer)
            notes = "only circular_outer_lit ok; annulus scatter unavailable"
    else:
        notes = "no circular annulus has both corrected mbar and sigma_meas"

    if method_inner != "not ok" or method_outer != "not ok":
        notes += f"; sigma_inner={method_inner}; sigma_outer={method_outer}"

    summary_rows.append({
        "kmin": float(kmin_value),
        "kmax": float(kmax_value),
        "mbar_inner_raw": mbar_inner_raw,
        "mbar_inner": mbar_inner,
        "sigma_inner": sigma_inner,
        "Pr_inner": float(row_inner.get("Pr", np.nan)) if row_inner is not None else np.nan,
        "Pr_over_P0_inner": float(row_inner.get("Pr_over_P0", np.nan)) if row_inner is not None else np.nan,
        "mbar_outer_raw": mbar_outer_raw,
        "mbar_outer": mbar_outer,
        "sigma_outer": sigma_outer,
        "Pr_outer": float(row_outer.get("Pr", np.nan)) if row_outer is not None else np.nan,
        "Pr_over_P0_outer": float(row_outer.get("Pr_over_P0", np.nan)) if row_outer is not None else np.nan,
        "mbar_weighted_raw": mbar_weighted_raw,
        "mbar_weighted": mbar_weighted,
        "sigma_weighted_formal": sigma_weighted_formal,
        "annulus_scatter": annulus_scatter,
        "sigma_adopted": sigma_adopted,
        "notes": notes,
    })

summary_columns = [
    "kmin",
    "kmax",
    "mbar_inner_raw",
    "mbar_inner",
    "sigma_inner",
    "Pr_inner",
    "Pr_over_P0_inner",
    "mbar_outer_raw",
    "mbar_outer",
    "sigma_outer",
    "Pr_outer",
    "Pr_over_P0_outer",
    "mbar_weighted_raw",
    "mbar_weighted",
    "sigma_weighted_formal",
    "annulus_scatter",
    "sigma_adopted",
    "notes",
]
df_annulus_summary = pd.DataFrame(summary_rows, columns=summary_columns)
display(df_annulus_summary)

recommendable = df_annulus_summary[
    np.isfinite(df_annulus_summary["mbar_weighted"])
    & np.isfinite(df_annulus_summary["sigma_adopted"])
].copy()

recommended_sbf = None
if recommendable.empty:
    print("[SBF-ANNULI] no recommended annulus-combined SBF result available")
else:
    main_kmin, main_kmax = FFT_K_RANGE_MAIN
    recommendable["is_main_window"] = (
        np.isclose(recommendable["kmin"].astype(float), float(main_kmin))
        & np.isclose(recommendable["kmax"].astype(float), float(main_kmax))
    )
    recommendable["uses_two_annuli"] = recommendable["notes"].str.contains(r"inner\+outer weighted", regex=True)
    recommendable = recommendable.sort_values(
        ["uses_two_annuli", "is_main_window", "sigma_adopted", "sigma_weighted_formal"],
        ascending=[False, False, True, True],
    )
    best = recommendable.iloc[0]
    recommended_sbf = best.to_dict()

    print(
        "[SBF-SCIENCE] recommended fixed-circular-annuli k-window "
        f"{best['kmin']:.3f}..{best['kmax']:.3f}"
    )
    print(f"[SBF-SCIENCE] weighted raw mbar       = {best['mbar_weighted_raw']:.4f} mag")
    print(f"[SBF-SCIENCE] weighted corrected mbar = {best['mbar_weighted']:.4f} mag")
    print(
        "[SBF-SCIENCE] formal={:.4f} mag, annulus_scatter={:.4f} mag, adopted={:.4f} mag".format(
            best["sigma_weighted_formal"],
            best["annulus_scatter"] if np.isfinite(best["annulus_scatter"]) else np.nan,
            best["sigma_adopted"],
        )
    )
    print(
        f"[SBF-SCIENCE] Pr/P0 inner={best['Pr_over_P0_inner']:.3%}, "
        f"outer={best['Pr_over_P0_outer']:.3%}"
    )
    print("[SBF-SCIENCE] adopted sigma = max(formal, annulus scatter); pipeline systematic is not measured yet")

pipeline_variants = pd.DataFrame([
    {
        "variant": "raw_current_residual",
        "status": "available",
        "mbar_spec": recommended_sbf["mbar_weighted"] if recommended_sbf is not None else np.nan,
        "notes": "science residual for circular annuli = sigma-capped resid_full_clip built from data - model_full; unresolved-source correction included via P_fluc = P0 - Pr",
    },
    {
        "variant": "sigma_clipped_or_capped_residual",
        "status": "not_run",
        "mbar_spec": np.nan,
        "notes": "future branch: rerun SBF after clipping/capping residual outliers",
    },
    {
        "variant": "spline_cleaned_residual",
        "status": "not_run",
        "mbar_spec": np.nan,
        "notes": "future branch: rerun SBF after spline-cleaned large-scale residuals",
    },
])

print("[SBF-PIPELINE] pipeline systematic placeholder:")
print("[SBF-PIPELINE] fill pipeline_variants with independent branch results, then use their mbar scatter as sigma_pipeline")
display(pipeline_variants)


[12:21:00] combining fixed circular SBF annuli with weighted mean (main science result)...


,kmin,kmax,mbar_inner_raw,mbar_inner,sigma_inner,Pr_inner,Pr_over_P0_inner,mbar_outer_raw,mbar_outer,sigma_outer,Pr_outer,Pr_over_P0_outer,mbar_weighted_raw,mbar_weighted,sigma_weighted_formal,annulus_scatter,sigma_adopted,notes
0,0.01,0.25,27.827420,27.840419,0.036878,0.204095,0.011901,27.634459,27.668027,0.037037,0.230142,0.030444,27.731356,27.754595,0.026133,0.086196,0.086196,inner+outer weighted; no pipeline systematic y...
1,0.03,0.25,27.919762,27.933923,0.036878,0.204095,0.012958,27.752252,27.789733,0.037037,0.230142,0.033933,27.836369,27.862140,0.026133,0.072095,0.072095,inner+outer weighted; no pipeline systematic y...
2,0.04,0.25,27.944310,27.958796,0.036878,0.204095,0.013254,27.776376,27.814715,0.037037,0.230142,0.034695,27.860705,27.887067,0.026133,0.072041,0.072041,inner+outer weighted; no pipeline systematic y...


[12:21:00] [SBF-SCIENCE] recommended fixed-circular-annuli k-window 0.040..0.250
[12:21:00] [SBF-SCIENCE] weighted raw mbar       = 27.8607 mag
[12:21:00] [SBF-SCIENCE] weighted corrected mbar = 27.8871 mag
[12:21:00] [SBF-SCIENCE] formal=0.0261 mag, annulus_scatter=0.0720 mag, adopted=0.0720 mag
[12:21:00] [SBF-SCIENCE] Pr/P0 inner=1.325%, outer=3.469%
[12:21:00] [SBF-SCIENCE] adopted sigma = max(formal, annulus scatter); pipeline systematic is not measured yet
[12:21:00] [SBF-PIPELINE] pipeline systematic placeholder:
[12:21:00] [SBF-PIPELINE] fill pipeline_variants with independent branch results, then use their mbar scatter as sigma_pipeline


,variant,status,mbar_spec,notes
0,raw_current_residual,available,27.887067,science residual for circular annuli = sigma-c...
1,sigma_clipped_or_capped_residual,not_run,NaN,future branch: rerun SBF after clipping/cappin...
2,spline_cleaned_residual,not_run,NaN,future branch: rerun SBF after spline-cleaned ...


## Цвета в тех же annuli

Измеряем F090W-F150W цвет в тех же круговых annuli, где измерялся SBF. Эти цвета являются диагностикой/калибровочным входом и не меняют уже измеренную F150W fluctuation power.


In [37]:
print("computing colors in the same fixed circular annuli used for SBF...")

required_annuli = ["circular_inner_lit", "circular_outer_lit"]
region_lookup = {reg["region"]: reg for reg in regions}
color_rows = []

if img_f090 is None:
    print("[COLOR-ANNULI] F090W image unavailable; annulus-matched colors skipped")
else:
    ny_c = min(img.shape[0], img_f090.shape[0])
    nx_c = min(img.shape[1], img_f090.shape[1])

    img150_c = img[:ny_c, :nx_c]
    img090_c = img_f090[:ny_c, :nx_c]
    valid150_c = valid150[:ny_c, :nx_c]
    valid090_c = valid090[:ny_c, :nx_c]
    premask_c = premask[:ny_c, :nx_c]

    for region_name in required_annuli:
        reg = region_lookup[region_name]
        reg_mask = reg["mask"][:ny_c, :nx_c]

        use_color = (
            reg_mask
            & (~premask_c)
            & valid150_c
            & valid090_c
            & np.isfinite(img150_c)
            & np.isfinite(img090_c)
        )

        n_raw = int(use_color.sum())
        row = {
            "region": region_name,
            "rin_arcsec": reg["rin_arcsec"],
            "rout_arcsec": reg["rout_arcsec"],
            "n_raw": n_raw,
            "n_clip": 0,
            "F150_med": np.nan,
            "F090_med": np.nan,
            "color_F090W_F150W": np.nan,
            "color_scatter": np.nan,
            "color_sem_proxy": np.nan,
            "notes": "",
        }

        if n_raw <= MIN_COLOR_PIXELS:
            row["notes"] = "too few usable pixels"
            color_rows.append(row)
            continue

        vals150 = img150_c[use_color]
        vals090 = img090_c[use_color]

        _, med150_clip, std150_clip = sigma_clipped_stats(vals150, sigma=COLOR_CLIP_SIGMA, maxiters=COLOR_CLIP_MAXIT)
        _, med090_clip, std090_clip = sigma_clipped_stats(vals090, sigma=COLOR_CLIP_SIGMA, maxiters=COLOR_CLIP_MAXIT)

        keep = (
            (vals150 >= med150_clip - COLOR_CLIP_SIGMA * std150_clip)
            & (vals150 <= med150_clip + COLOR_CLIP_SIGMA * std150_clip)
            & (vals090 >= med090_clip - COLOR_CLIP_SIGMA * std090_clip)
            & (vals090 <= med090_clip + COLOR_CLIP_SIGMA * std090_clip)
        )

        vals150_rob = vals150[keep]
        vals090_rob = vals090[keep]
        positive = (vals150_rob > 0.0) & (vals090_rob > 0.0)
        vals150_rob = vals150_rob[positive]
        vals090_rob = vals090_rob[positive]
        n_clip = int(vals150_rob.size)

        row["n_clip"] = n_clip
        if n_clip <= MIN_COLOR_PIXELS:
            row["notes"] = "too few positive sigma-clipped pixels"
            color_rows.append(row)
            continue

        f150_med = float(np.nanmedian(vals150_rob))
        f090_med = float(np.nanmedian(vals090_rob))
        color = float(-2.5 * np.log10(f090_med / f150_med))

        pixel_colors = -2.5 * np.log10(vals090_rob / vals150_rob)
        color_med = float(np.nanmedian(pixel_colors))
        color_mad = float(1.4826 * np.nanmedian(np.abs(pixel_colors - color_med)))
        color_sem_proxy = float(color_mad / np.sqrt(n_clip)) if np.isfinite(color_mad) else np.nan

        row.update({
            "F150_med": f150_med,
            "F090_med": f090_med,
            "color_F090W_F150W": color,
            "color_scatter": color_mad,
            "color_sem_proxy": color_sem_proxy,
            "notes": "same circular annulus as SBF; scatter is pixel-color diagnostic, not full calibration error",
        })
        color_rows.append(row)

df_color_annuli = pd.DataFrame(color_rows)
display(df_color_annuli)

color_summary_rows = []
if not df_color_annuli.empty:
    ok_color = df_color_annuli[
        df_color_annuli["region"].isin(required_annuli)
        & np.isfinite(df_color_annuli["color_F090W_F150W"])
    ].copy()

    if len(ok_color) == 2 and recommended_sbf is not None:
        sigma_inner = float(recommended_sbf.get("sigma_inner", np.nan))
        sigma_outer = float(recommended_sbf.get("sigma_outer", np.nan))
        if _finite_positive(sigma_inner) and _finite_positive(sigma_outer):
            weights_by_region = {
                "circular_inner_lit": 1.0 / sigma_inner**2,
                "circular_outer_lit": 1.0 / sigma_outer**2,
            }
            weights = ok_color["region"].map(weights_by_region).to_numpy(dtype=float)
            colors = ok_color["color_F090W_F150W"].to_numpy(dtype=float)
            color_weighted = float(np.sum(weights * colors) / np.sum(weights))
            color_annulus_scatter = float(0.5 * (np.nanmax(colors) - np.nanmin(colors)))
            color_summary_rows.append({
                "summary": "SBF-weighted circular-annuli color",
                "color_F090W_F150W": color_weighted,
                "sigma_proxy": color_annulus_scatter,
                "notes": "uses same annulus weights as recommended SBF; sigma_proxy is half inner-outer color difference, not full calibration error",
            })

    if not ok_color.empty:
        color_summary_rows.append({
            "summary": "mean of available circular-annuli colors",
            "color_F090W_F150W": float(np.nanmean(ok_color["color_F090W_F150W"])),
            "sigma_proxy": float(np.nanstd(ok_color["color_F090W_F150W"], ddof=1)) if len(ok_color) > 1 else np.nan,
            "notes": "simple annulus-matched diagnostic color summary",
        })

df_color_summary = pd.DataFrame(color_summary_rows)
display(df_color_summary)


[12:21:00] computing colors in the same fixed circular annuli used for SBF...


,region,rin_arcsec,rout_arcsec,n_raw,n_clip,F150_med,F090_med,color_F090W_F150W,color_scatter,color_sem_proxy,notes
0,circular_inner_lit,8.2,16.4,649088,636008,13.651158,8.796837,0.477107,0.047410,0.000059,same circular annulus as SBF; scatter is pixel...
1,circular_outer_lit,16.4,32.8,2554434,2493567,4.235677,3.073884,0.348088,0.115095,0.000073,same circular annulus as SBF; scatter is pixel...


,summary,color_F090W_F150W,sigma_proxy,notes
0,SBF-weighted circular-annuli color,0.412876,0.06451,uses same annulus weights as recommended SBF; ...
1,mean of available circular-annuli colors,0.412598,0.09123,simple annulus-matched diagnostic color summary
